## Experimental contract

```text
Dataset:      google/fleurs (8 Indic languages), language-balanced pool re-sampled every epoch (3000 instances / language / epoch)
Model:        openai/whisper-small (pinned commit); d_model 768, 12 heads, tokenizer, vocabulary, positional encodings, embeddings unchanged
Trainable:    encoder layers 7-12 (FFN 768-6144-768) + encoder final LayerNorm + decoder layers 7-12 (FFN 768-21504-768) + decoder final LayerNorm   (~298 M parameters)
Frozen:       conv front-end, positional embeddings, encoder layers 1-6, decoder layers 1-6, token embedding / tied output projection (all in eval() mode)
Widening:     function-preserving: pretrained FFN kept in the first 3072 neurons, new neurons start with zero output weights
Training:     2 epochs, AdamW(0.9, 0.98), wd 0.01, LR 2e-5, 120 warm-up steps (capped at 20 % of the run), cosine to 10 %, grad-clip 1.0, FP16,
              effective batch 32 (micro-batch chosen from the memory estimate), SpecAugment (training only)
Selection:    fixed validation subset every 250 steps -> best 2 checkpoints -> full validation set picks the final one; early stopping on stagnation
Decoding:     greedy, forced language + task=transcribe, no timestamps, max_new_tokens 440
Test set:     never used for optimisation, checkpoint selection, early stopping or hyper-parameter decisions; decoded only after the final checkpoint is chosen on validation.
              Reported test results = the filtered, cap-compatible subset produced by the fixed preprocessing pipeline.
Primary:      WER        Secondary: CER (+ script accuracy, failure rate)
```

No auxiliary losses and no extra heads. The tied output projection and the token embeddings stay frozen.

In [ ]:
import os, sys, time, shutil, importlib, importlib.util, importlib.metadata as md, subprocess
NOTEBOOK_T0 = time.time()

ON_KAGGLE = os.path.isdir("/kaggle")
OUT_DIR = "/kaggle/working" if ON_KAGGLE else os.path.abspath("./outputs")
SCRATCH = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else ("/tmp" if ON_KAGGLE else os.path.abspath("./scratch"))
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(SCRATCH, exist_ok=True)
# big dataset caches go to scratch space (not the 20 GB /kaggle/working quota); must be set before `import datasets`
os.environ.setdefault("HF_DATASETS_CACHE", f"{SCRATCH}/hf_datasets")
os.environ.setdefault("HF_HUB_CACHE", f"{SCRATCH}/hf_hub")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

REQUIRED = {
    "torch": None, "transformers": "transformers", "datasets": "datasets", "accelerate": "accelerate",
    "evaluate": "evaluate", "jiwer": "jiwer", "soundfile": "soundfile", "librosa": "librosa",
    "numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "seaborn": "seaborn",
    "scipy": "scipy", "safetensors": "safetensors", "rapidfuzz": "rapidfuzz", "tqdm": "tqdm",
    "huggingface_hub": "huggingface_hub",
}
missing = [pip for mod, pip in REQUIRED.items() if pip and importlib.util.find_spec(mod) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("All required packages already present - nothing to install.")

for mod in REQUIRED:
    try: v = md.version(mod.replace("_", "-") if mod in ("huggingface_hub",) else mod)
    except Exception: v = getattr(importlib.import_module(mod), "__version__", "?")
    print(f"{mod:16s} {v}")
print(f"\nOUT_DIR={OUT_DIR}  SCRATCH={SCRATCH}")
print("HF_DATASETS_CACHE =", os.environ["HF_DATASETS_CACHE"])

In [ ]:
import random, numpy as np, torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
# Deterministic cuDNN selection. We deliberately do NOT call torch.use_deterministic_algorithms(True):
# some SDPA / scatter kernels have no deterministic variants that are fast on a T4.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

assert torch.cuda.is_available(), "This notebook targets a CUDA GPU (Kaggle T4). Enable the GPU accelerator."
DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU  : {GPU_NAME}")
print(f"VRAM : {GPU_VRAM_GB:.2f} GiB")
print(f"CUDA : {torch.version.cuda}   torch {torch.__version__}")
print("Reproducibility: seeded + deterministic cuDNN selection, but NOT bitwise deterministic (index_add_ / SDPA kernels).")

In [ ]:
import json, math, re, io, copy, inspect, unicodedata, itertools, contextlib, warnings, gc, random, shutil, hashlib
from collections import Counter, defaultdict
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import soundfile as sf
import jiwer
from rapidfuzz.distance import Levenshtein
from scipy.stats import spearmanr
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import datasets
from datasets import load_dataset, Audio
import transformers
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from huggingface_hub import HfApi
from safetensors.torch import save_file, load_file
from functools import lru_cache

@lru_cache(maxsize=None)
def is_latin_letter(ch):
    return ch.isalpha() and "LATIN" in unicodedata.name(ch, "")

CFG = dict(
    MODEL_ID="openai/whisper-small",
    MODEL_REVISION_PIN="973afd24965f72e36ca33b3055d56a652f456b4d",
    DATASET_ID="google/fleurs",
    LANG_CONFIGS={"bn": "bn_in", "gu": "gu_in", "hi": "hi_in", "kn": "kn_in",
                  "ml": "ml_in", "mr": "mr_in", "ta": "ta_in", "te": "te_in"},
    LANG_NAMES={"bn": "Bengali", "gu": "Gujarati", "hi": "Hindi", "kn": "Kannada",
                "ml": "Malayalam", "mr": "Marathi", "ta": "Tamil", "te": "Telugu"},
    # Unicode block (inclusive code-point range) of the expected script for each language
    LANG_SCRIPT={"bn": ("Bengali", 0x0980, 0x09FF), "gu": ("Gujarati", 0x0A80, 0x0AFF),
                 "hi": ("Devanagari", 0x0900, 0x097F), "kn": ("Kannada", 0x0C80, 0x0CFF),
                 "ml": ("Malayalam", 0x0D00, 0x0D7F), "mr": ("Devanagari", 0x0900, 0x097F),
                 "ta": ("Tamil", 0x0B80, 0x0BFF), "te": ("Telugu", 0x0C00, 0x0C7F)},
    TEXT_FIELD="raw_transcription",
    SEED=SEED, SAMPLE_RATE=16000, MIN_AUDIO_SEC=1.0, MAX_AUDIO_SEC=30.0,
    MAX_TARGET_TOKENS=440,
    N_PREFIX=4,
    # --- architecture (0-based layer indices: 6..11 = layers 7..12) ---
    TRAIN_ENC_LAYERS=list(range(6, 12)), TRAIN_DEC_LAYERS=list(range(6, 12)),
    ENC_FFN_WIDTH=6144, DEC_FFN_WIDTH=21504,
    TARGET_MICRO_BS=4,
    MEM_FRACTION_LIMIT=0.80,
    SPECAUGMENT=dict(enabled=True, time_prob=0.05, time_length=10, time_min_masks=2, feature_prob=0.05, feature_length=10, feature_min_masks=0),
    BASE_LR=2e-5, WARMUP_STEPS=120, WARMUP_MAX_FRACTION=0.20, MIN_LR_RATIO=0.10, WEIGHT_DECAY=0.01,
    BETAS=(0.9, 0.98), GRAD_CLIP=1.0, NUM_EPOCHS=2, TARGET_EFFECTIVE_BATCH=32, NUM_WORKERS=3, TRAIN_PER_LANG=3000,
    MAX_CONSEC_NONFINITE=3, MAX_CONSEC_SKIPPED=10,
    CKPT_TOPK=2, LOG_EVERY_STEPS=10, EVAL_EVERY_STEPS=250, EARLY_STOP_PATIENCE=4, VAL_SUBSET_PER_LANG=25, EVAL_BATCH=32, MAX_NEW_TOKENS=440,
    BOOTSTRAP_B=1000, SPEED_UTTS_PER_LANG=16, SCRIPT_OK_THRESHOLD=0.5,      # >= 50 % of letters in the expected script => predominant-script hit
    PREVIOUS_DENSE_198M=dict(trainable_params=198_406_688, total_params=373_990_688, test_wer=50.51, test_cer=14.37, script_acc=99.97, failure_rate=3.56, train_minutes=47.0,
                             note="baseline Dense-198M (single run; per-utterance outputs are not available, so no paired test)"),
    LICENSES=dict(source="Hugging Face Hub metadata (nothing is assumed)", base_model=None, dataset=None),
    RECIPE="function-preserving scaling; enc 7-12 (FFN 6144) + dec 7-12 (FFN 21504) trainable; SpecAugment; 2 epochs with per-epoch balanced resampling; best-checkpoint selection",
    OUT_DIR=OUT_DIR,
)
TAG = "dense"
LANGS = list(CFG["LANG_CONFIGS"].keys())
LANG_NAME = CFG["LANG_NAMES"]
PREFIX_LANG_TOKENS = {l: f"<|{l}|>" for l in LANGS}

def resolve_model_revision():
    """Strict: the experiment is pinned to one exact Whisper-small commit. If it cannot be resolved the run STOPS (no silent fallback to `main`)."""
    try:
        return HfApi().model_info(CFG["MODEL_ID"], revision=CFG["MODEL_REVISION_PIN"]).sha, "pinned (strict)"
    except Exception as e:
        raise RuntimeError(f"Pinned revision {CFG['MODEL_REVISION_PIN']} of {CFG['MODEL_ID']} could not be resolved ({type(e).__name__}: {str(e)[:120]}). "
                           "Refusing to fall back to the latest revision. Check that Internet is enabled, or change MODEL_REVISION_PIN deliberately.") from e

MODEL_REVISION, REV_NOTE = resolve_model_revision()
CFG["MODEL_REVISION"] = MODEL_REVISION
def hub_license(fetch, *args, **kw):
    """License exactly as declared in the Hub metadata (card_data.license, else a `license:` tag); None if it cannot be read. Never guessed."""
    try:
        info = fetch(*args, **kw); lic = getattr(getattr(info, "card_data", None), "license", None)
        if lic is None: lic = next((t.split(":", 1)[1] for t in (getattr(info, "tags", None) or []) if str(t).startswith("license:")), None)
        return lic
    except Exception: return None
_NA = "not available from the Hub metadata - check the model / dataset card"
CFG["LICENSES"]["base_model"] = hub_license(HfApi().model_info, CFG["MODEL_ID"], revision=MODEL_REVISION) or _NA
CFG["LICENSES"]["dataset"] = hub_license(HfApi().dataset_info, CFG["DATASET_ID"]) or _NA

print("=" * 70)
print(f"Model name        : {CFG['MODEL_ID']}")
print(f"Model revision    : {MODEL_REVISION}  [{REV_NOTE}]")
print(f"Transformers      : {transformers.__version__}")
print(f"Datasets          : {datasets.__version__}")
print(f"PyTorch           : {torch.__version__}")
print(f"CUDA              : {torch.version.cuda}")
print(f"Licenses (Hub)    : model = {CFG['LICENSES']['base_model']} | dataset = {CFG['LICENSES']['dataset']}")
print(f"GPU name          : {GPU_NAME}")
print(f"GPU memory        : {GPU_VRAM_GB:.2f} GiB")
print("=" * 70)

def _log(*a): print(f"[{(time.time()-NOTEBOOK_T0)/60:7.1f} min]", *a, flush=True)

In [ ]:
def free_hub_cache():
    """The prepared Arrow cache is self-contained, so the downloaded parquet copies can be deleted to save disk."""
    try:
        from huggingface_hub import scan_cache_dir
        info = scan_cache_dir()
        revs = [r.commit_hash for repo in info.repos if repo.repo_id == CFG["DATASET_ID"] and repo.repo_type == "dataset" for r in repo.revisions]
        if revs:
            st = info.delete_revisions(*revs); st.execute()
            _log(f"freed {st.expected_freed_size/1e9:.1f} GB of downloaded parquet from the hub cache")
        dl = os.path.join(os.environ["HF_DATASETS_CACHE"], "downloads")
        if os.path.isdir(dl):
            gb = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(dl) for f in fs) / 1e9
            shutil.rmtree(dl, ignore_errors=True); _log(f"freed {gb:.1f} GB from {dl}")
    except Exception as e:
        print("  (hub-cache cleanup skipped:", type(e).__name__, str(e)[:80], ")")

def dir_gb(path):
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(path) for f in fs) / 1e9 if os.path.isdir(path) else 0.0

def disk_report(tag=""):
    du = shutil.disk_usage(SCRATCH)
    _log(f"disk[{tag}] scratch free {du.free/1e9:6.1f} GB | datasets cache {dir_gb(os.environ['HF_DATASETS_CACHE']):5.1f} GB | hub cache {dir_gb(os.environ['HF_HUB_CACHE']):5.1f} GB")

def load_fleurs_split(split):
    out = {}
    for lang, cfg_name in CFG["LANG_CONFIGS"].items():
        t0 = time.time()
        assert shutil.disk_usage(SCRATCH).free > 6e9, "less than 6 GB of scratch disk left - stopping before the FLEURS download exhausts the disk"
        ds = load_dataset(CFG["DATASET_ID"], cfg_name, split=split)
        # decode=False: keep the encoded bytes. Decoding happens lazily in the data loader (soundfile), so the whole
        # corpus is never materialised in RAM and nothing depends on torchcodec/ffmpeg (datasets>=4 audio backend).
        ds = ds.cast_column("audio", Audio(sampling_rate=CFG["SAMPLE_RATE"], decode=False))
        ds = ds.add_column("lang", [lang] * len(ds))
        out[lang] = ds
        _log(f"{split:10s} {lang} ({cfg_name}): {len(ds):5d} rows in {time.time()-t0:5.0f}s")
        if all(c["filename"].endswith(".arrow") for c in ds.cache_files): free_hub_cache()
        else: print("  NOTE: dataset is not Arrow-backed in this `datasets` version -> downloaded files are kept")
        disk_report(f"{split}/{lang}")
    return out

train_ds = load_fleurs_split("train")
print("columns:", train_ds["bn"].column_names)

In [ ]:
val_ds = load_fleurs_split("validation")

In [ ]:
test_ds = load_fleurs_split("test")
disk_report("after all splits")

In [ ]:
SPLITS = {"train": train_ds, "validation": val_ds, "test": test_ds}
rows = []
for lang in LANGS:
    r = {"lang": lang, "language": LANG_NAME[lang]}
    for s, d in SPLITS.items(): r[f"{s}_count"] = len(d[lang])
    ex = train_ds[lang][0]
    r["audio_feature"] = f"{type(train_ds[lang].features['audio']).__name__}(sr={train_ds[lang].features['audio'].sampling_rate}, decode={train_ds[lang].features['audio'].decode})"
    r["language_field(ex0)"] = ex.get("language"); r["lang_id(ex0)"] = ex.get("lang_id")
    r["gender_values"] = ",".join(map(str, sorted(set(train_ds[lang]["gender"])))) if "gender" in train_ds[lang].column_names else "n/a"
    r["has_speaker_id"] = any(c in train_ds[lang].column_names for c in ("speaker_id", "speaker"))
    rows.append(r)
insp = pd.DataFrame(rows)
display(insp)
ex = train_ds["hi"][0]
print("\nExample fields (hi, train[0]):", {k: (v if k != "audio" else f"<{len(v['bytes'] or b'')} bytes, path={v['path']}>") for k, v in ex.items()})
print("\nTranscription field in use:", CFG["TEXT_FIELD"], "| 'transcription' = normalised (lower-cased, no punctuation); 'raw_transcription' = original text.")
print("Speaker metadata: FLEURS exposes `gender` only; no speaker id column. Train vs dev/test speaker separation is a guarantee of the FLEURS dataset card; it is NOT verified here (no speaker ids).")

## Audio processing
Audio is validated from the *encoded header* (sample rate, channels, frames) without decoding the waveform, so RAM stays flat. During training/evaluation the waveform is decoded lazily per batch, down-mixed to mono and resampled to 16 kHz if necessary.

In [ ]:
def _audio_src(a):
    if a is None: return None
    if a.get("bytes"): return io.BytesIO(a["bytes"])
    p = a.get("path")
    return p if (p and os.path.exists(p)) else None

def scan_split(ds, lang, split):
    recs = []
    ids = ds["id"]     # NOTE: FLEURS `id` is a SENTENCE id shared by up to 3 speaker recordings -> NOT unique per utterance; utterance id = split + row + sentence id
    txt = ds[CFG["TEXT_FIELD"]] if CFG["TEXT_FIELD"] in ds.column_names else ds["transcription"]
    i = 0
    for batch in tqdm(ds.select_columns(["audio"]).iter(batch_size=128), total=math.ceil(len(ds)/128), desc=f"scan {split}/{lang}", leave=False):
        for a in batch["audio"]:
            sr = ch = fr = None; ok = False
            src = _audio_src(a)
            if src is not None:
                try:
                    inf = sf.info(src); sr, ch, fr, ok = inf.samplerate, inf.channels, inf.frames, True
                except Exception: ok = False
            recs.append(dict(split=split, lang=lang, row=i, sent_id=ids[i], id=f"fleurs-{lang}-{split}-r{i}-s{ids[i]}", raw_text=txt[i],
                             sr=sr, channels=ch, frames=fr, audio_ok=ok, dur=(fr / sr) if ok and sr else np.nan))
            i += 1
    return recs

all_recs = []
for split, d in SPLITS.items():
    for lang in LANGS: all_recs += scan_split(d[lang], lang, split)
meta = pd.DataFrame(all_recs)
meta["text_ok"] = meta["raw_text"].map(lambda t: isinstance(t, str) and len(t.strip()) > 0)

audit = meta.groupby(["split", "lang"]).agg(n=("id", "size"), missing_audio=("audio_ok", lambda s: int((~s).sum())),
        missing_text=("text_ok", lambda s: int((~s).sum())), sr_values=("sr", lambda s: ",".join(map(str, sorted(set(s.dropna().astype(int)))))),
        channel_values=("channels", lambda s: ",".join(map(str, sorted(set(s.dropna().astype(int)))))), min_dur=("dur", "min"), max_dur=("dur", "max")).reset_index()
display(audit)
print("Non-16 kHz clips :", int((meta.sr.dropna() != CFG["SAMPLE_RATE"]).sum()), "(resampled on the fly)")
print("Non-mono clips   :", int((meta.channels.dropna() != 1).sum()), "(down-mixed on the fly)")

In [ ]:
keep_flags = (meta.audio_ok & meta.text_ok & (meta.dur >= CFG["MIN_AUDIO_SEC"]) & (meta.dur <= CFG["MAX_AUDIO_SEC"]))
meta["keep_dur"] = keep_flags
dur_report = (meta.groupby(["split", "lang"]).agg(original=("id", "size"), remaining=("keep_dur", "sum")).reset_index())
dur_report["removed"] = dur_report.original - dur_report.remaining
dur_report["removal_%"] = (100 * dur_report.removed / dur_report.original).round(2)
display(dur_report[["split", "lang", "original", "removed", "remaining", "removal_%"]])
print("Total removed by audio/text/duration validity:", int(dur_report.removed.sum()), "of", int(dur_report.original.sum()))

## Transcript processing
Training/target text: Unicode **NFC**, whitespace collapsed, stripped. Punctuation is **kept** here; evaluation normalisation (which removes it) is applied separately and identically to reference and hypothesis.

In [ ]:
def clean_transcript(t):
    t = unicodedata.normalize("NFC", t if isinstance(t, str) else "")
    return re.sub(r"\s+", " ", t).strip()

meta["text"] = meta["raw_text"].map(clean_transcript)
print("Examples:"); display(meta[meta.split == "train"].groupby("lang").head(1)[["lang", "text"]].reset_index(drop=True))
print("Empty after cleaning:", int((meta.text == "").sum()))

In [ ]:
REV_KW = dict(revision=MODEL_REVISION)
processor = WhisperProcessor.from_pretrained(CFG["MODEL_ID"], **REV_KW)
tokenizer, feature_extractor = processor.tokenizer, processor.feature_extractor
EOS_ID = tokenizer.eos_token_id
def encode_text(t):
    return tokenizer(t, add_special_tokens=False).input_ids

m = meta[meta.keep_dur & (meta.text != "")].copy()
m["text_ids"] = m.text.map(encode_text)
m["n_target"] = m.text_ids.map(len) + CFG["N_PREFIX"] + 1
m["keep_tok"] = m.n_target <= CFG["MAX_TARGET_TOKENS"]
tok_report = m.groupby(["split", "lang"]).agg(before=("id", "size"), kept=("keep_tok", "sum"), mean_tokens=("n_target", "mean"), max_tokens=("n_target", "max")).reset_index()
tok_report["removed"] = tok_report.before - tok_report.kept
tok_report["removal_%"] = (100 * tok_report.removed / tok_report.before).round(2)
display(tok_report.round(1))
meta_final = m[m.keep_tok].reset_index(drop=True)
print(f"Final examples: {len(meta_final)} (removed for token length: {int((~m.keep_tok).sum())})")
heavy = tok_report[tok_report["removal_%"] > 5]
if len(heavy):
    print("NOTE - the 440-token cap removes >5 % of these (split, language) groups; Whisper's tokenizer needs many tokens per Indic character, so the cap is NOT language-neutral.")
    print("       Reported test WER/CER therefore covers the shorter-sentence subset of those languages (utterances beyond the 448-position decoder cannot be transcribed anyway):")
    display(heavy[["split", "lang", "before", "removed", "removal_%"]])

In [ ]:
# FLEURS stores several speaker recordings per sentence; those are different audio and are all kept. Only an exact duplicate recording
# (same language, same transcript, same number of audio frames) counts as a duplicate. The validation and test sets are never modified here.
m["dup_key"] = m.lang + "|" + m.text + "|" + m.frames.astype("Int64").astype(str)
m["is_dup"] = m.duplicated(subset=["split", "dup_key"], keep="first")
m["keep_dedup"] = m.keep_tok & ~(m.is_dup & (m.split == "train"))

_ev = m[m.split.isin(["validation", "test"])]
_eval_texts, _eval_sents = {(r.lang, r.text) for r in _ev.itertuples()}, {(r.lang, str(r.sent_id)) for r in _ev.itertuples()}
m["leaks_to_eval"] = (m.split == "train") & [((l, t) in _eval_texts) or ((l, str(s)) in _eval_sents) for l, t, s in zip(m.lang, m.text, m.sent_id)]
m["keep_dedup"] = m.keep_dedup & ~m.leaks_to_eval

dedup_report = m.groupby(["split", "lang"]).agg(before=("keep_tok", "sum"), exact_duplicates=("is_dup", "sum"), train_eval_text_overlap=("leaks_to_eval", "sum"),
                                                 kept=("keep_dedup", "sum")).reset_index()
display(dedup_report)
meta_final = m[m.keep_dedup].reset_index(drop=True)
print(f"Final examples after de-duplication: {len(meta_final)}  (training: {int((meta_final.split == 'train').sum())}, validation: {int((meta_final.split == 'validation').sum())}, test: {int((meta_final.split == 'test').sum())})")
print("Validation / test duplicates are only reported (those sets are never filtered beyond the shared audio / transcript / token-length rules):",
      int(m[m.split != "train"].is_dup.sum()))
assert not ((meta_final.split == "train") & meta_final.leaks_to_eval).any(), "train / eval text leakage after filtering"

## Multilingual training pool
The pool is a table of *references* (language, row index) into the per-language Arrow datasets - the audio is **not copied or duplicated**. Each pool entry carries `id`, `lang`, `transcription` and resolves its `audio` lazily.

In [ ]:
class FleursPool(torch.utils.data.Dataset):
    """Reference table over per-language HF datasets. Item = dict(id, lang, audio=<16 kHz mono float32>, transcription, text_ids)."""
    def __init__(self, table, hf_by_lang):
        self.hf = hf_by_lang
        self.lang = table.lang.tolist(); self.row = table.row.tolist(); self.ids = table.id.tolist()
        self.text = table.text.tolist(); self.text_ids = table.text_ids.tolist(); self.dur = table.dur.tolist()
    def __len__(self): return len(self.ids)
    def load_audio(self, i):
        a = self.hf[self.lang[i]][self.row[i]]["audio"]
        wav, sr = sf.read(_audio_src(a), dtype="float32", always_2d=True)
        wav = wav.mean(axis=1)
        if sr != CFG["SAMPLE_RATE"]:
            import librosa; wav = librosa.resample(wav, orig_sr=sr, target_sr=CFG["SAMPLE_RATE"])
        return wav
    def __getitem__(self, i):
        return dict(id=self.ids[i], lang=self.lang[i], audio=self.load_audio(i), transcription=self.text[i], text_ids=self.text_ids[i])

def balance_language_sampling(table, per_lang, seed):
    """Exactly `per_lang` training rows per language, deterministic. A language with fewer utterances is covered by whole repeats plus a random
    remainder (no utterance is repeated more often than needed); a language with more is sub-sampled WITHOUT replacement."""
    rs = np.random.RandomState(seed); parts = []
    for l in LANGS:
        t = table[table.lang == l].reset_index(drop=True); n = len(t)
        reps, rem = divmod(per_lang, n)
        idx = list(range(n)) * reps + (sorted(rs.choice(n, size=rem, replace=False).tolist()) if rem else [])
        parts.append(t.iloc[idx])
    return pd.concat(parts, ignore_index=True)

tables = {s: meta_final[meta_final.split == s].reset_index(drop=True) for s in SPLITS}
tables["train_available"] = tables["train"]
# Across epochs each language is re-sampled deterministically (seed SEED + epoch); languages with more than TRAIN_PER_LANG usable utterances may expose different subsets in different epochs
# (not every available utterance is necessarily used), and every language contributes the same number of instances. The training loader reads the concatenated pool once, sequentially.
epoch_blocks = [balance_language_sampling(tables["train_available"], CFG["TRAIN_PER_LANG"], SEED + e).sample(frac=1.0, random_state=SEED + e).reset_index(drop=True)
                for e in range(CFG["NUM_EPOCHS"])]
tables["train"] = pd.concat(epoch_blocks, ignore_index=True)
train_pool = FleursPool(tables["train"], train_ds)
val_pool   = FleursPool(tables["validation"], val_ds)
test_pool  = FleursPool(tables["test"], test_ds)
print({k: len(v) for k, v in dict(train=train_pool, validation=val_pool, test=test_pool).items()})
print("first 40 pool languages:", tables["train"].lang.head(40).tolist())
longest_run = max(len(list(g)) for _, g in itertools.groupby(tables["train"].lang))
print("longest contiguous single-language run in the shuffled pool:", longest_run)
assert (tables["train"].groupby("lang").size() == CFG["TRAIN_PER_LANG"] * CFG["NUM_EPOCHS"]).all(), "training pool is not language-balanced"

In [ ]:
def stats_for(key):
    t = tables[key]
    return t.groupby("lang").agg(n=("id", "size"), unique=("id", "nunique"), hours=("dur", lambda s: s.sum() / 3600), mean_s=("dur", "mean"), median_s=("dur", "median"))
st_av, st_tr, st_va, st_te = (stats_for(k) for k in ("train_available", "train", "validation", "test"))
stat_table = pd.DataFrame({
    "Language": [LANG_NAME[l] for l in LANGS],
    "Unique clean train utterances (FLEURS, after filters)": [st_av.loc[l, "n"] for l in LANGS],
    "Training instances (all epochs)": [st_tr.loc[l, "n"] for l in LANGS],
    "Unique utterances used": [st_tr.loc[l, "unique"] for l in LANGS],
    "Repeat factor (instances / unique available)": [round(st_tr.loc[l, "n"] / st_av.loc[l, "n"], 2) for l in LANGS],
    "Validation examples": [st_va.loc[l, "n"] for l in LANGS],
    "Test examples": [st_te.loc[l, "n"] for l in LANGS],
    "Train hours (all epochs)": [round(st_tr.loc[l, "hours"], 2) for l in LANGS],
    "Unique train hours": [round(st_av.loc[l, "hours"], 2) for l in LANGS],
    "Train mean dur (s)": [round(st_tr.loc[l, "mean_s"], 2) for l in LANGS],
    "Train median dur (s)": [round(st_tr.loc[l, "median_s"], 2) for l in LANGS]}, index=LANGS)
display(stat_table)
TOTAL_TRAIN_HOURS = float(st_tr.hours.sum())
print(f"unique clean training utterances available : {int(st_av.n.sum())}  ({float(st_av.hours.sum()):.1f} h)")
print(f"training instances (all {CFG['NUM_EPOCHS']} epochs)           : {len(tables['train'])}   (unique used: {int(st_tr.unique.sum())})  -> {len(tables['train']) / int(st_av.n.sum()):.2f} passes over the unique data")
per_lang_n = {LANG_NAME[l]: int(st_tr.loc[l, "n"]) for l in LANGS}
print("instances per language                    :", per_lang_n)
print(f"validation examples                       : {len(tables['validation'])}   test examples: {len(tables['test'])}")
lang_share = {LANG_NAME[l]: round(100 * st_tr.loc[l, "n"] / st_tr.n.sum(), 1) for l in LANGS}
print("language distribution (% of training pool):", lang_share)
print(f"\nDATASET-SIZE DECISION: FLEURS has only {int(st_av.n.sum())} unique clean training utterances ({float(st_av.hours.sum()):.0f} h) after the 1-30 s and 440-token filters. That is FEWER than the "
      f"{CFG['TRAIN_PER_LANG'] * len(LANGS)} instances of one balanced epoch, so no larger clean training pool exists inside FLEURS and repetition cannot be avoided if the languages are to stay balanced. "
      f"Each of the {CFG['NUM_EPOCHS']} epochs therefore draws its own balanced sample of {CFG['TRAIN_PER_LANG']} instances per language (largest repeat factor within one epoch "
      f"{float((CFG['TRAIN_PER_LANG'] / st_av.n).max()):.2f}x); repeated instances are NOT new data and are reported as such.")
for s_ in ("train_available", "validation", "test"):
    t_ = tables[s_]; n_s = t_.groupby('lang').sent_id.nunique().sum()
    print(f"{s_:16s}: {len(t_)} utterances of {n_s} distinct (language, sentence) pairs -> {len(t_)/n_s:.2f} speaker recordings per sentence")
assert tables['train_available'].id.is_unique and tables['validation'].id.is_unique and tables['test'].id.is_unique, 'utterance ids must be unique'
assert set(tables['train'].id) <= set(tables['train_available'].id), "training pool contains utterances outside the training split"
assert not (set(tables['train'].id) & set(tables['test'].id)) and not (set(tables['train'].id) & set(tables['validation'].id)), "train / validation / test must be disjoint"

In [ ]:
def funnel_for(split):
    rows = []
    for l in LANGS:
        mm = meta[(meta.split == split) & (meta.lang == l)]
        a_ok = mm[mm.audio_ok & (mm.dur >= CFG["MIN_AUDIO_SEC"]) & (mm.dur <= CFG["MAX_AUDIO_SEC"])]
        t_ok = a_ok[a_ok.text_ok & (a_ok.text != "")]
        mt = m[(m.split == split) & (m.lang == l)]
        assert len(mt) == len(t_ok)
        r = dict(language=LANG_NAME[l], raw_examples=len(mm), after_audio_filter=len(a_ok), after_transcript_filter=len(t_ok),
                 after_token_length_filter=int(mt.keep_tok.sum()), after_dedup=int(mt.keep_dedup.sum()))
        r["final_usable_unique"] = r["after_dedup"]
        if split == "train":
            tt = tables["train"][tables["train"].lang == l]
            r["selected_training_instances"] = len(tt); r["unique_instances_used"] = int(tt.id.nunique()); r["repeated_instances"] = len(tt) - r["unique_instances_used"]
            eb = epoch_blocks[0][epoch_blocks[0].lang == l]; r["per_epoch_unique"] = int(eb.id.nunique()); r["per_epoch_repeated"] = len(eb) - r["per_epoch_unique"]
            r["per_epoch_sampling"] = "sub-sampled without replacement" if r["final_usable_unique"] >= CFG["TRAIN_PER_LANG"] else "deterministic repetition (whole repeats + random remainder)"
            r["max_repeat_count"] = int(tt.groupby("id").size().max()); r["repeat_factor"] = round(len(tt) / max(1, r["final_usable_unique"]), 2)
        rows.append(r)
    return pd.DataFrame(rows).set_index("language")
FUNNEL = {s: funnel_for(s) for s in ("train", "validation", "test")}
for s in FUNNEL:
    print(f"\n--- {s}: data funnel per language ---"); display(FUNNEL[s])
print("\n'selected_training_instances' counts every instance fed to the optimiser over all epochs; 'unique_instances_used' counts distinct utterances; 'repeated_instances' = selected - unique (repeats are NOT new data); 'per_epoch_*' describe one balanced epoch block.")
print("The 440-token cap removes a language-dependent share of utterances (Whisper needs many tokens per Indic character), so reported WER/CER refer to the cap-compatible subset of each language.")

In [ ]:
LEAK_SETS = {"train": tables["train_available"], "validation": tables["validation"], "test": tables["test"]}
LEAKAGE_REPORT = {}
for a_, b_ in (("train", "validation"), ("train", "test"), ("validation", "test")):
    sent_a, sent_b = set(zip(LEAK_SETS[a_].lang, LEAK_SETS[a_].sent_id.astype(str))), set(zip(LEAK_SETS[b_].lang, LEAK_SETS[b_].sent_id.astype(str)))
    text_a, text_b = set(zip(LEAK_SETS[a_].lang, LEAK_SETS[a_].text)), set(zip(LEAK_SETS[b_].lang, LEAK_SETS[b_].text))
    LEAKAGE_REPORT[f"{a_} & {b_}"] = dict(shared_sentence_ids=len(sent_a & sent_b), shared_transcripts=len(text_a & text_b))
_pool_txt = set(zip(tables["train"].lang, tables["train"].text)); _pool_sent = set(zip(tables["train"].lang, tables["train"].sent_id.astype(str)))
for b_ in ("validation", "test"):
    LEAKAGE_REPORT[f"training pool & {b_}"] = dict(shared_sentence_ids=len(_pool_sent & set(zip(LEAK_SETS[b_].lang, LEAK_SETS[b_].sent_id.astype(str)))),
                                                   shared_transcripts=len(_pool_txt & set(zip(LEAK_SETS[b_].lang, LEAK_SETS[b_].text))))
display(pd.DataFrame(LEAKAGE_REPORT).T)
for k_, r_ in LEAKAGE_REPORT.items():
    assert r_["shared_sentence_ids"] == 0, f"LEAKAGE: {k_} share {r_['shared_sentence_ids']} FLEURS sentence ids"
    if k_.startswith(("train", "training")): assert r_["shared_transcripts"] == 0, f"LEAKAGE: {k_} share {r_['shared_transcripts']} transcripts"
    elif r_["shared_transcripts"]: print(f"WARNING: {k_} share {r_['shared_transcripts']} identical transcripts under different sentence ids (neither split is used for training, so this is not training leakage)")
print("LEAKAGE CHECK PASSED: train / validation / test are pairwise disjoint at sentence level; the test set is never used for training, checkpoint selection, early stopping or any hyper-parameter.")

In [ ]:
print("Sampling rate (feature extractor):", feature_extractor.sampling_rate)
print("Feature extractor config        :", {k: v for k, v in feature_extractor.to_dict().items() if k in ("feature_size", "n_fft", "hop_length", "chunk_length", "n_samples", "nb_max_frames", "padding_value")})
print("Tokenizer                       :", type(tokenizer).__name__, "| vocab size", len(tokenizer))
SOT_ID = tokenizer.convert_tokens_to_ids("<|startoftranscript|>")
TRANSCRIBE_ID = tokenizer.convert_tokens_to_ids("<|transcribe|>")
NOTIMESTAMPS_ID = tokenizer.convert_tokens_to_ids("<|notimestamps|>")
PAD_ID = tokenizer.pad_token_id
print("decoder start token (SOT)       :", SOT_ID, tokenizer.convert_ids_to_tokens(SOT_ID))
print("EOS token                       :", tokenizer.eos_token_id, tokenizer.eos_token)
print("PAD token                       :", PAD_ID, tokenizer.pad_token)
assert tokenizer.eos_token_id == PAD_ID == 50257, "EOS and PAD are expected to share id 50257 -> label masking MUST use lengths"
print("OK: EOS == PAD == 50257  (padding is therefore masked by sequence length, never by `labels == 50257`)")
_special = set(tokenizer.all_special_ids) | set(tokenizer.added_tokens_encoder.values())
assert min(_special) == tokenizer.eos_token_id, "token-id layout assumption violated"
print("OK: every id < EOS is an ordinary text token; EOS, language/task/notimestamps prefix and timestamp tokens are all >= EOS "
      "-> `(ids < EOS_ID)` counts transcript text tokens only (used for n_gen_tokens and tokens/sec)")

In [ ]:
def load_base_model():
    m = WhisperForConditionalGeneration.from_pretrained(CFG["MODEL_ID"], **REV_KW)
    m.config.forced_decoder_ids = None; m.generation_config.forced_decoder_ids = None   # language/task are passed explicitly
    m.config.use_cache = True
    m.generation_config.max_length = None              # we always pass max_new_tokens=440; avoids the max_length/max_new_tokens clash warning
    return m

t0 = time.time()
base_cpu = load_base_model().eval()
print(f"loaded in {time.time()-t0:.1f}s | attn implementation: {getattr(base_cpu.config, '_attn_implementation', '?')}")
c = base_cpu.config
arch = dict(encoder_layers=c.encoder_layers, decoder_layers=c.decoder_layers, d_model=c.d_model,
            decoder_ffn_dim=c.decoder_ffn_dim, encoder_ffn_dim=c.encoder_ffn_dim,
            decoder_attention_heads=c.decoder_attention_heads, encoder_attention_heads=c.encoder_attention_heads,
            activation_function=c.activation_function, dropout=c.dropout, activation_dropout=c.activation_dropout,
            max_target_positions=c.max_target_positions)
print(json.dumps(arch, indent=1))
assert (c.encoder_layers, c.decoder_layers, c.d_model, c.decoder_ffn_dim, c.decoder_attention_heads) == (12, 12, 768, 3072, 12)
enc, dec = base_cpu.model.encoder, base_cpu.model.decoder
assert len(enc.layers) == 12 and len(dec.layers) == 12
print("encoder children :", [n for n, _ in enc.named_children()])
print("decoder children :", [n for n, _ in dec.named_children()])
print("decoder layer 0  :", [n for n, _ in dec.layers[0].named_children()])
print("output projection tied to token embedding:", base_cpu.proj_out.weight.data_ptr() == dec.embed_tokens.weight.data_ptr())

In [ ]:
def configure_trainable(model):
    for p in model.parameters(): p.requires_grad_(False)
    enc, dec = model.model.encoder, model.model.decoder
    for i in CFG["TRAIN_ENC_LAYERS"]:
        for p in enc.layers[i].parameters(): p.requires_grad_(True)
    for i in CFG["TRAIN_DEC_LAYERS"]:
        for p in dec.layers[i].parameters(): p.requires_grad_(True)
    for p in enc.layer_norm.parameters(): p.requires_grad_(True)
    for p in dec.layer_norm.parameters(): p.requires_grad_(True)
    return model

def set_train_mode(model):
    """model.train() for the trainable parts; every module whose parameters are ALL frozen is put back into eval() so that
    frozen layers never run training-time behaviour (dropout etc.), independent of the current config values.
    (WhisperModel itself stays in train mode - it contains trainable layers - which is what activates SpecAugment on the input features.)"""
    model.train()
    for m in model.modules():
        ps = list(m.parameters())
        if ps and not any(p.requires_grad for p in ps): m.eval()
    return model

def count_params(model):
    tot = sum(p.numel() for p in model.parameters()); tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return dict(total=tot, trainable=tr, frozen=tot - tr)

def trainable_prefixes():
    return tuple([f"model.encoder.layers.{i}." for i in CFG["TRAIN_ENC_LAYERS"]] + [f"model.decoder.layers.{i}." for i in CFG["TRAIN_DEC_LAYERS"]] +
                 ["model.encoder.layer_norm.", "model.decoder.layer_norm."])

def check_trainable_names(model):
    allowed = trainable_prefixes()
    bad = [n for n, p in model.named_parameters() if p.requires_grad and not n.startswith(allowed)]
    leaked = [n for n, p in model.named_parameters() if (not p.requires_grad) and n.startswith(allowed)]
    assert not bad, f"unexpected trainable params: {bad[:5]}"
    assert not leaked, f"params that should be trainable are frozen: {leaked[:5]}"
    return True

_demo = configure_trainable(copy.deepcopy(base_cpu)); check_trainable_names(_demo); set_train_mode(_demo)
_de, _dd = _demo.model.encoder, _demo.model.decoder
n_frozen_enc, n_frozen_dec = min(CFG["TRAIN_ENC_LAYERS"]), min(CFG["TRAIN_DEC_LAYERS"])
assert not any(_de.layers[i].training for i in range(n_frozen_enc)) and all(_de.layers[i].training for i in CFG["TRAIN_ENC_LAYERS"])
assert not any(_dd.layers[i].training for i in range(n_frozen_dec)) and all(_dd.layers[i].training for i in CFG["TRAIN_DEC_LAYERS"])
assert not _dd.embed_tokens.training and not _de.conv1.training and _dd.layer_norm.training and _de.layer_norm.training and _demo.model.training
print(f"train/eval modes: frozen encoder layers 1-{n_frozen_enc}, decoder layers 1-{n_frozen_dec}, embeddings and conv front-end are in eval(); encoder layers {n_frozen_enc + 1}-12, decoder layers {n_frozen_dec + 1}-12 and both final LayerNorms are in train()")
del _de, _dd
print("Unmodified pretrained layout with the same trainable layers (reference for the parameter arithmetic only) ->", {k: f"{v/1e6:.2f} M" for k, v in count_params(_demo).items()})
del _demo

## Function-preserving widening of the FFNs
A Whisper encoder or decoder layer computes `FFN(x) = fc2(dropout(act(fc1(LN(x)))))` (verified from the installed source in Cell 24). Widening replaces `fc1: 768 -> 3072` and `fc2: 3072 -> 768` by `768 -> W` and `W -> 768`:
* neurons `0..3071` are the pretrained FFN, **unchanged**;
* the new neurons `3072..W-1` have `fc1` rows ~ N(0, std of the pretrained `fc1` weights), bias 0, and **`fc2` columns = 0**, so the widened FFN computes *exactly* the pretrained function at initialisation while the new neurons are distinct (random) and receive gradients through `fc2`.

No pretrained layer is re-initialised. Plain replication of the pretrained FFN is avoided on purpose: identical replicas receive identical gradients and would never diverge.

In [ ]:
for kind, layer in (("encoder", base_cpu.model.encoder.layers[0]), ("decoder", base_cpu.model.decoder.layers[0])):
    src_ = inspect.getsource(type(layer).forward)
    for tok in ["self.final_layer_norm(", "self.fc1(", "self.activation_fn(", "self.fc2(", "activation_dropout"]:
        assert tok in src_, f"expected `{tok}` in {type(layer).__name__}.forward"
    pos = [src_.index(t) for t in ("self.final_layer_norm(", "self.fc1(", "self.fc2(")]
    assert pos == sorted(pos), "unexpected FFN execution order"
    print(f"{type(layer).__name__}.forward - FFN sub-block as implemented in transformers {transformers.__version__}:\n")
    print("\n".join(src_[src_.rindex("residual = hidden_states"):].splitlines()[:10]))
    print("attributes:", {a: type(getattr(layer, a)).__name__ for a in ("fc1", "fc2", "activation_fn", "final_layer_norm")}, "| activation_dropout =", layer.activation_dropout, "| dropout =", layer.dropout, "\n")
assert base_cpu.config.encoder_ffn_dim == base_cpu.config.decoder_ffn_dim == 3072
print("=> FFN(x) = fc2(dropout(activation_fn(fc1(final_layer_norm(x))))) added to the residual, in both stacks. Widening fc1 / fc2 only therefore changes the FFN width.")

In [ ]:
D_MODEL, H0 = base_cpu.config.d_model, base_cpu.config.decoder_ffn_dim
def ffn_param_count(width): return 2 * D_MODEL * width + width + D_MODEL                  # fc1 + fc2 incl. biases
REF_TRAINABLE = sum(p.numel() for n, p in base_cpu.named_parameters() if n.startswith(trainable_prefixes()))      # trainable params of the unmodified layout (arithmetic reference only)
N_ENC_T, N_DEC_T = len(CFG["TRAIN_ENC_LAYERS"]), len(CFG["TRAIN_DEC_LAYERS"])
def trainable_analytic(w_enc, w_dec): return REF_TRAINABLE + N_ENC_T * (ffn_param_count(w_enc) - ffn_param_count(H0)) + N_DEC_T * (ffn_param_count(w_dec) - ffn_param_count(H0))
def total_analytic(w_enc, w_dec): return sum(p.numel() for p in base_cpu.parameters()) + N_ENC_T * (ffn_param_count(w_enc) - ffn_param_count(H0)) + N_DEC_T * (ffn_param_count(w_dec) - ffn_param_count(H0))
ENC_FFN_WIDTH, DEC_FFN_WIDTH = CFG["ENC_FFN_WIDTH"], CFG["DEC_FFN_WIDTH"]
print("exact parameter counts for candidate (encoder FFN, decoder FFN) widths of the trainable layers:")
for we, wd in ((H0, 25600), (6144, 24576), (6144, 22528), (6144, 21504), (6144, 18432), (9216, 18432), (12288, 15360)):
    print(f"  enc {we:5d} ({we / H0:3.1f}x) / dec {wd:5d} ({wd / H0:4.1f}x): trainable {trainable_analytic(we, wd)/1e6:7.2f} M | total {total_analytic(we, wd)/1e6:7.2f} M" + ("   <== selected" if (we, wd) == (ENC_FFN_WIDTH, DEC_FFN_WIDTH) else ""))
EXPECTED_TRAINABLE_PARAMS, EXPECTED_TOTAL_PARAMS = trainable_analytic(ENC_FFN_WIDTH, DEC_FFN_WIDTH), total_analytic(ENC_FFN_WIDTH, DEC_FFN_WIDTH)
print(f"architecture specification -> expected trainable parameters {EXPECTED_TRAINABLE_PARAMS:,} ({EXPECTED_TRAINABLE_PARAMS / 1e6:.2f} M) | expected total {EXPECTED_TOTAL_PARAMS:,} ({EXPECTED_TOTAL_PARAMS / 1e6:.2f} M)")

def widen_ffn_pair(fc1, fc2, width, gen):
    """Function-preserving widening  768 -> h0 -> 768  ==>  768 -> width -> 768 (see the markdown above)."""
    h0, d = fc1.out_features, fc1.in_features
    assert width >= h0
    new1, new2 = nn.Linear(d, width), nn.Linear(width, d)
    with torch.no_grad():
        new1.weight[:h0] = fc1.weight.cpu(); new1.bias[:h0] = fc1.bias.cpu()
        new1.weight[h0:] = torch.randn(width - h0, d, generator=gen) * float(fc1.weight.std()); new1.bias[h0:] = 0.0
        new2.weight[:, :h0] = fc2.weight.cpu(); new2.weight[:, h0:] = 0.0; new2.bias.copy_(fc2.bias.cpu())
    return new1.to(fc1.weight.device), new2.to(fc1.weight.device)

def widen_ffns(model, enc_width, dec_width):
    gen = torch.Generator().manual_seed(CFG["SEED"] + 1)
    for stack, ids, width in ((model.model.encoder, CFG["TRAIN_ENC_LAYERS"], enc_width), (model.model.decoder, CFG["TRAIN_DEC_LAYERS"], dec_width)):
        for i in ids:
            L = stack.layers[i]
            assert isinstance(L.fc1, nn.Linear) and L.fc1.out_features == H0, "layer already modified"
            L.fc1, L.fc2 = widen_ffn_pair(L.fc1, L.fc2, width, gen)
    return model

def apply_specaugment_config(model):
    """Whisper's built-in SpecAugment (WhisperModel._mask_input_features): masks time spans / mel bands of the INPUT FEATURES, only in training mode."""
    sa = CFG["SPECAUGMENT"]
    model.config.apply_spec_augment = bool(sa["enabled"])
    model.config.mask_time_prob, model.config.mask_time_length, model.config.mask_time_min_masks = sa["time_prob"], sa["time_length"], sa["time_min_masks"]
    model.config.mask_feature_prob, model.config.mask_feature_length, model.config.mask_feature_min_masks = sa["feature_prob"], sa["feature_length"], sa["feature_min_masks"]
    return model

def build_model():
    torch.manual_seed(CFG["SEED"])
    m = widen_ffns(copy.deepcopy(base_cpu), ENC_FFN_WIDTH, DEC_FFN_WIDTH); m.generation_config.max_length = None
    apply_specaugment_config(m)
    return set_train_mode(configure_trainable(m))

In [ ]:
torch.manual_seed(CFG["SEED"])
x = torch.randn(4, 37, D_MODEL, device=DEVICE)
ffn_results = {}
for kind, layer, width in (("decoder", base_cpu.model.decoder.layers[CFG["TRAIN_DEC_LAYERS"][0]], DEC_FFN_WIDTH), ("encoder", base_cpu.model.encoder.layers[CFG["TRAIN_ENC_LAYERS"][0]], ENC_FFN_WIDTH)):
    fc1_o, fc2_o, act = copy.deepcopy(layer.fc1).to(DEVICE), copy.deepcopy(layer.fc2).to(DEVICE), layer.activation_fn
    with torch.no_grad():
        ref = fc2_o(act(fc1_o(x)))
        w1_, w2_ = widen_ffn_pair(fc1_o, fc2_o, width, torch.Generator().manual_seed(CFG["SEED"] + 1))
        out_wide = w2_(act(w1_(x)))
    mx = (out_wide - ref).abs().max().item(); rl = ((out_wide - ref).norm() / ref.norm()).item(); ok_ = (mx < 1e-4) and (rl < 1e-5)
    ffn_results[kind] = dict(max_abs_err=mx, relative_err=rl, passed=bool(ok_))
    print("(%s) single %s FFN : max |err| = %.3e | relative err = %.3e   (widened %d -> %d neurons)  -> %s" % ("a" if kind == "decoder" else "b", kind, mx, rl, H0, width, "PASS" if ok_ else "FAIL"))
    del fc1_o, fc2_o, w1_, w2_, out_wide, ref

ref_net = copy.deepcopy(base_cpu).to(DEVICE).eval(); wide_net = build_model().to(DEVICE).eval()
feats = torch.randn(2, feature_extractor.feature_size, 3000, device=DEVICE) * 0.5
dec_in = torch.tensor([[SOT_ID, tokenizer.convert_tokens_to_ids("<|hi|>"), TRANSCRIBE_ID, NOTIMESTAMPS_ID] + [1200, 345, 6789, 4321, 77, 910, 2222, 3333]] * 2, device=DEVICE)
with torch.no_grad():
    lg_ref = ref_net(input_features=feats, decoder_input_ids=dec_in).logits
    lg_wide = wide_net(input_features=feats, decoder_input_ids=dec_in).logits
net_max = (lg_ref - lg_wide).abs().max().item(); net_rel = ((lg_ref - lg_wide).norm() / lg_ref.norm()).item()
net_pass = net_max < 2e-3 and net_rel < 1e-4
print("(c) whole model (encoder + decoder widened): max |logit err| = %.3e | relative err = %.3e (logit scale %.1f)  -> %s" % (net_max, net_rel, lg_ref.abs().max().item(), "PASS" if net_pass else "FAIL"))

wm = wide_net.model; f0 = torch.randn(8, 80, 3000, device=DEVICE) + 3.0
wm.eval(); same_eval = torch.equal(wm._mask_input_features(f0.clone()), f0)
wm.train(); torch.manual_seed(0); np.random.seed(0); f1 = wm._mask_input_features(f0.clone()); frac_t = float(((f1 == 0).all(dim=1)).float().mean()); frac_f = float(((f1 == 0).all(dim=2)).float().mean())
sa_pass = same_eval and (not torch.equal(f1, f0)) and 0.005 < frac_t < 0.15
print("(d) SpecAugment : eval mode leaves features untouched = %s | train mode masks %.1f %% of time frames and %.1f %% of mel bins (configured ~5 %%)  -> %s" % (same_eval, 100 * frac_t, 100 * frac_f, "PASS" if sa_pass else "FAIL"))
EQUIV_RESULT = dict(ffn=ffn_results, model_logit_max_abs_err=net_max, model_logit_relative_err=net_rel, specaugment_time_masked_pct=100 * frac_t, specaugment_mel_masked_pct=100 * frac_f,
                    passed=bool(all(r["passed"] for r in ffn_results.values()) and net_pass and sa_pass))
del ref_net, wide_net, lg_ref, lg_wide, feats, x, f0, f1; gc.collect(); torch.cuda.empty_cache()
if not EQUIV_RESULT["passed"]:
    raise AssertionError("Initialisation-equivalence / SpecAugment test FAILED - fix the model construction before training.")
print("\nINITIALISATION EQUIVALENCE TEST (function-preserving widening, encoder + decoder) + SPECAUGMENT CHECK: PASS")

In [ ]:
model = build_model()
check_trainable_names(model)
pc = count_params(model); PARAM_COUNTS = dict(model=pc)
MODEL_NAME = f"Dense-{round(pc['trainable'] / 1e6)}M"
print(f"{MODEL_NAME}\n  total parameters     : {pc['total']:,}\n  trainable parameters : {pc['trainable']:,}\n  frozen parameters    : {pc['frozen']:,}")
assert pc["trainable"] == EXPECTED_TRAINABLE_PARAMS and pc["total"] == EXPECTED_TOTAL_PARAMS, f"instantiated model ({pc['trainable']:,} trainable / {pc['total']:,} total) does not match the architecture specification ({EXPECTED_TRAINABLE_PARAMS:,} / {EXPECTED_TOTAL_PARAMS:,})"
print(f"  trainable fraction   : {100 * pc['trainable'] / pc['total']:.2f} % of all parameters")
prev = CFG["PREVIOUS_DENSE_198M"]
print(f"  vs baseline Dense-198M ({prev['trainable_params']:,} trainable): x{pc['trainable'] / prev['trainable_params']:.2f}")

dec_layers, enc_layers = model.model.decoder.layers, model.model.encoder.layers
for stack_name, layers, base_layers, train_ids, width in (("decoder", dec_layers, base_cpu.model.decoder.layers, CFG["TRAIN_DEC_LAYERS"], DEC_FFN_WIDTH), ("encoder", enc_layers, base_cpu.model.encoder.layers, CFG["TRAIN_ENC_LAYERS"], ENC_FFN_WIDTH)):
    for i in range(12):
        Lo, Ld = base_layers[i], layers[i]
        if i in train_ids:
            assert isinstance(Ld.fc1, nn.Linear) and (Ld.fc1.in_features, Ld.fc1.out_features) == (D_MODEL, width) and (Ld.fc2.in_features, Ld.fc2.out_features) == (width, D_MODEL)
            assert torch.equal(Ld.fc1.weight[:H0], Lo.fc1.weight) and torch.equal(Ld.fc1.bias[:H0], Lo.fc1.bias) and torch.equal(Ld.fc2.weight[:, :H0], Lo.fc2.weight) and torch.equal(Ld.fc2.bias, Lo.fc2.bias)
            assert float(Ld.fc2.weight[:, H0:].abs().max()) == 0.0 and not torch.equal(Ld.fc1.weight[H0:H0 + 8], Ld.fc1.weight[H0 + 8:H0 + 16])
        else:
            assert Ld.fc1.out_features == H0 and torch.equal(Ld.fc1.weight, Lo.fc1.weight)
_bp = dict(base_cpu.named_parameters())
assert all(torch.equal(p, _bp[n]) for n, p in model.named_parameters() if n in _bp and p.shape == _bp[n].shape), "a shared parameter differs from the pretrained checkpoint"
assert model.proj_out.weight.data_ptr() == model.model.decoder.embed_tokens.weight.data_ptr() and not model.proj_out.weight.requires_grad, "output projection must stay tied and frozen"
print(f"architecture OK: encoder layers {[i + 1 for i in CFG['TRAIN_ENC_LAYERS']]} FFN {D_MODEL}->{ENC_FFN_WIDTH}->{D_MODEL} | decoder layers {[i + 1 for i in CFG['TRAIN_DEC_LAYERS']]} FFN {D_MODEL}->{DEC_FFN_WIDTH}->{D_MODEL} (function-preserving); all other layers unchanged; tied output projection frozen")
print("initialisation OK: pretrained neurons bit-identical, new neurons' output weights == 0, every other weight identical to the pretrained checkpoint")
print(f"\ntrainable : encoder layers {[i + 1 for i in CFG['TRAIN_ENC_LAYERS']]} + encoder final LayerNorm, decoder layers {[i + 1 for i in CFG['TRAIN_DEC_LAYERS']]} + decoder final LayerNorm")
print("frozen    : conv front-end, positional embeddings, encoder layers 1-%d, decoder layers 1-%d, token embedding / tied output projection (all in eval() mode)" % (min(CFG["TRAIN_ENC_LAYERS"]), min(CFG["TRAIN_DEC_LAYERS"])))
display(pd.DataFrame({"": ["Total Params", "Trainable Params", "Frozen Params"], MODEL_NAME: [pc["total"], pc["trainable"], pc["frozen"]]}).set_index(""))

# ---- analytic GPU-memory estimate; the micro-batch is derived from it (the real peak is logged during training) ----
def estimate_memory_gb(micro, worst_tokens=440):
    """static (GiB): fp32 weights + fp16 weight copies + fp32 grads + AdamW m,v.  activations: worst case (440 target tokens) saved tensors of the trainable layers
    (FFN: fc1 output, GELU input, GELU output) + the frozen decoder layers on the gradient path + fp16/fp32 logits and their gradient."""
    static = (4 * pc["total"] + 2 * pc["total"] + 4 * pc["trainable"] + 8 * pc["trainable"]) / 1024**3
    per_sample = (N_ENC_T * (28e6 + 3 * 1500 * ENC_FFN_WIDTH * 2) + N_DEC_T * (3 * worst_tokens * DEC_FFN_WIDTH * 2 + 11e6) + (12 - N_DEC_T) * 20e6 + worst_tokens * base_cpu.config.vocab_size * 10) / 1024**3
    return static, per_sample * micro
MEM_BUDGET_GB = CFG["MEM_FRACTION_LIMIT"] * GPU_VRAM_GB
for mb in (1, 2, 4):
    s_, a_ = estimate_memory_gb(mb); print(f"  estimated GPU memory, micro-batch {mb}: static {s_:.1f} GiB + activations (worst case) {a_:.1f} GiB = {s_ + a_:.1f} GiB   (budget {MEM_BUDGET_GB:.1f} GiB = {CFG['MEM_FRACTION_LIMIT']:.0%} of {GPU_VRAM_GB:.1f} GiB)")
SAFE_MICRO_BS = max([mb for mb in (1, 2, 4) if mb <= CFG["TARGET_MICRO_BS"] and sum(estimate_memory_gb(mb)) <= MEM_BUDGET_GB] or [1])
print(f"largest micro-batch whose worst-case estimate stays within the budget: {SAFE_MICRO_BS}")

## Collator, conditioning, loss, optimiser and schedule
Cells 28-33. The collator uses `PREFIX_IDS` (defined in Cell 29) at call time.

In [ ]:
class ASRCollator:
    """Training / teacher-forcing collator.
    sequence = <SOT><lang><transcribe><notimestamps> text... <EOS>;  decoder_input = seq[:-1], target = seq[1:].
    Padding is identified by the TRUE sequence lengths (EOS == PAD == 50257, so `labels == 50257` is never used)."""
    def __init__(self, feature_extractor): self.fe = feature_extractor
    def __call__(self, batch):
        feats = self.fe([b["audio"] for b in batch], sampling_rate=CFG["SAMPLE_RATE"], return_tensors="pt").input_features
        seqs = [PREFIX_IDS[b["lang"]] + list(b["text_ids"]) + [EOS_ID] for b in batch]
        lens = torch.tensor([len(s) for s in seqs])
        B, T = len(seqs), int(lens.max())
        full = torch.full((B, T), PAD_ID, dtype=torch.long)
        for i, s in enumerate(seqs): full[i, :len(s)] = torch.tensor(s)
        dec_in, targets = full[:, :-1].contiguous(), full[:, 1:].contiguous()
        pos = torch.arange(T - 1)[None, :]
        valid = pos < (lens - 1)[:, None]                          # real decoder-input positions (not padding)
        loss_mask = valid & (pos >= CFG["N_PREFIX"] - 1)           # predictions of text tokens + EOS only (no conditioning prefix)
        labels = targets.masked_fill(~loss_mask, -100)
        return dict(input_features=feats, decoder_input_ids=dec_in, labels=labels, valid=valid, lens=lens,
                    ids=[b["id"] for b in batch], langs=[b["lang"] for b in batch], refs=[b["transcription"] for b in batch])

def eval_collate(batch):
    """Inference collator: features + metadata only (no targets)."""
    feats = feature_extractor([b["audio"] for b in batch], sampling_rate=CFG["SAMPLE_RATE"], return_tensors="pt").input_features
    return dict(input_features=feats, ids=[b["id"] for b in batch], langs=[b["lang"] for b in batch],
                refs=[b["transcription"] for b in batch], audio_sec=[len(b["audio"]) / CFG["SAMPLE_RATE"] for b in batch],
                text_ids=[b["text_ids"] for b in batch])

collator = ASRCollator(feature_extractor)

In [ ]:
LANG_TOKEN_IDS = {l: tokenizer.convert_tokens_to_ids(PREFIX_LANG_TOKENS[l]) for l in LANGS}
assert all(i is not None and i != tokenizer.unk_token_id for i in LANG_TOKEN_IDS.values())
PREFIX_IDS = {l: [SOT_ID, LANG_TOKEN_IDS[l], TRANSCRIBE_ID, NOTIMESTAMPS_ID] for l in LANGS}
assert all(len(v) == CFG["N_PREFIX"] for v in PREFIX_IDS.values())
rows = []
for l in LANGS:
    tokenizer.set_prefix_tokens(language=l, task="transcribe", predict_timestamps=False)
    ids = tokenizer("x").input_ids
    assert ids[:4] == PREFIX_IDS[l] and ids[-1] == EOS_ID, (l, ids)
    rows.append(dict(lang=l, language=LANG_NAME[l], lang_token=PREFIX_LANG_TOKENS[l], lang_id=LANG_TOKEN_IDS[l], prefix=PREFIX_IDS[l]))
display(pd.DataFrame(rows))
print("SOT", SOT_ID, "| transcribe", TRANSCRIBE_ID, "| notimestamps", NOTIMESTAMPS_ID, "| EOS", EOS_ID)
print("All prefixes verified against tokenizer.set_prefix_tokens(language, task='transcribe', predict_timestamps=False).")
_b = collator([train_pool[0], train_pool[1]])
print("collator shapes:", {k: tuple(v.shape) for k, v in _b.items() if torch.is_tensor(v)})
print("labels row0 (first 8):", _b["labels"][0, :8].tolist(), "... text/EOS targets start at position 3")
_n0 = int((_b["labels"][0] != -100).sum()); print("loss tokens row0:", _n0, "= len(text_ids)+1 =", len(train_pool[0]["text_ids"]) + 1)
assert _n0 == len(train_pool[0]["text_ids"]) + 1
del _b
# ---- loss alignment on REAL collator output (EOS == PAD == 50257, so the mask must come from lengths): shortest, median and longest transcripts of the first 400 pool entries ----
assert len(set(LANG_TOKEN_IDS.values())) == len(LANGS) == 8, "the eight languages need eight distinct language tokens"
def _probe_item(i): return dict(id=train_pool.ids[i], lang=train_pool.lang[i], audio=np.zeros(CFG["SAMPLE_RATE"], dtype=np.float32), transcription=train_pool.text[i], text_ids=train_pool.text_ids[i])
_order = sorted(range(min(len(train_pool), 400)), key=lambda i: len(train_pool.text_ids[i])); _probe = [_order[0], _order[len(_order) // 2], _order[-1], 0]
_pb = collator([_probe_item(i) for i in _probe])
for r_, i_ in enumerate(_probe):
    txt_ids = list(train_pool.text_ids[i_]); pre_ = PREFIX_IDS[train_pool.lang[i_]]; seq_ = pre_ + txt_ids + [EOS_ID]; L_ = len(seq_)
    lab_, din_, val_ = _pb["labels"][r_], _pb["decoder_input_ids"][r_], _pb["valid"][r_]
    assert din_[:L_ - 1].tolist() == seq_[:-1], "decoder input must be the sequence without its last token"
    tgt_pos = (lab_ != -100).nonzero().flatten().tolist()
    assert tgt_pos == list(range(CFG["N_PREFIX"] - 1, L_ - 1)), "loss positions must be exactly those predicting transcript tokens + EOS"
    assert lab_[tgt_pos].tolist() == txt_ids + [EOS_ID], "label[j] must equal sequence[j + 1] (shifted alignment)"
    assert int(lab_[L_ - 2]) == EOS_ID, "the real EOS must be retained as the last target"
    assert not (set(lab_[tgt_pos].tolist()) & set(pre_)), "prefix tokens must not be loss targets"
    assert (lab_[:CFG["N_PREFIX"] - 1] == -100).all() and (lab_[L_ - 1:] == -100).all(), "prefix predictions and padded positions must be masked"
    assert int(val_.sum()) == L_ - 1 and (din_[L_ - 1:] == PAD_ID).all(), "padded decoder positions must be real padding"
print("LOSS ALIGNMENT VERIFIED on", len(_probe), "collated sequences: targets = transcript tokens + EOS, EOS kept although EOS == PAD, prefix and padding excluded")
LOSS_ALIGNMENT_OK = True
del _pb

In [ ]:
# CELL 30 - ASR loss: causal cross-entropy over transcript tokens + genuine EOS; only true padding / prefix masked
def asr_loss(logits, labels, reduction="mean"):
    return F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), labels.reshape(-1), ignore_index=-100, reduction=reduction)

def run_forward(model, batch):
    """Teacher-forced forward pass (SpecAugment is applied inside WhisperModel when - and only when - the model is in training mode)."""
    feats = batch["input_features"].to(DEVICE, non_blocking=True)
    dec_in = batch["decoder_input_ids"].to(DEVICE, non_blocking=True)
    return model(input_features=feats, decoder_input_ids=dec_in, use_cache=False)
print("loss = token-mean cross-entropy over text tokens + EOS (conditioning prefix and true padding masked by sequence length, never by `label == 50257`)")

In [ ]:
def build_optimizer(model):
    params = [p for p in model.parameters() if p.requires_grad]
    return torch.optim.AdamW([dict(params=params, lr=CFG["BASE_LR"], weight_decay=CFG["WEIGHT_DECAY"], name="all_trainable")], betas=CFG["BETAS"])

_o = build_optimizer(model)
print({g["name"]: (len(g["params"]), sum(p.numel() for p in g["params"]), g["lr"]) for g in _o.param_groups}, "(group: (#tensors, #params, lr))")
assert sum(p.numel() for g in _o.param_groups for p in g["params"]) == PARAM_COUNTS["model"]["trainable"]
del _o

In [ ]:
# CELL 32 - LR schedule: linear warm-up (120 steps) -> cosine decay to 10 % of the peak LR
def make_scheduler(opt, total_steps):
    w, mr = CFG["WARMUP_STEPS"], CFG["MIN_LR_RATIO"]
    def factor(step):
        if step < w: return (step + 1) / w
        prog = min(1.0, (step - w) / max(1, total_steps - w))
        return mr + (1 - mr) * 0.5 * (1 + math.cos(math.pi * prog))
    return torch.optim.lr_scheduler.LambdaLR(opt, factor)

In [ ]:
MICRO_BS = SAFE_MICRO_BS
ACCUM = max(1, CFG["TARGET_EFFECTIVE_BATCH"] // MICRO_BS)
EFFECTIVE_BS = MICRO_BS * ACCUM
N_TRAIN = len(train_pool)
MICRO_PER_RUN = N_TRAIN // MICRO_BS                       # drop_last
TOTAL_STEPS = MICRO_PER_RUN // ACCUM                      # the pool already contains all epochs (one balanced block per epoch)
STEPS_PER_EPOCH = TOTAL_STEPS / CFG["NUM_EPOCHS"]
WARMUP_STEPS = min(CFG["WARMUP_STEPS"], max(10, int(CFG["WARMUP_MAX_FRACTION"] * TOTAL_STEPS)))     # 120 steps, but never more than 20 % of the run
if WARMUP_STEPS < CFG["WARMUP_STEPS"]:
    print(f"WARNING: the training set gives only {TOTAL_STEPS} optimiser steps -> warm-up capped from {CFG['WARMUP_STEPS']} to {WARMUP_STEPS} steps")
CFG["WARMUP_STEPS_SPEC"] = CFG["WARMUP_STEPS"]; CFG["WARMUP_STEPS"] = WARMUP_STEPS
assert TOTAL_STEPS > WARMUP_STEPS, "fewer optimisation steps than warm-up steps"
DROPPED_TRAIN = N_TRAIN - TOTAL_STEPS * EFFECTIVE_BS
CFG.update(MICRO_BS=MICRO_BS, ACCUM=ACCUM, EFFECTIVE_BS=EFFECTIVE_BS, TOTAL_STEPS=TOTAL_STEPS, N_TRAIN=N_TRAIN)
EVAL_STEPS = sorted({s for s in range(CFG["EVAL_EVERY_STEPS"], TOTAL_STEPS, CFG["EVAL_EVERY_STEPS"])} | {int(round(STEPS_PER_EPOCH * e)) for e in range(1, CFG["NUM_EPOCHS"])} | {TOTAL_STEPS})
_ids_md5 = lambda ids: hashlib.md5("\n".join(ids).encode("utf-8")).hexdigest()
FINGERPRINT = dict(
    model=CFG["MODEL_ID"], revision=MODEL_REVISION, dataset=CFG["DATASET_ID"], configs=CFG["LANG_CONFIGS"], text_field=CFG["TEXT_FIELD"],
    audio_sec=[CFG["MIN_AUDIO_SEC"], CFG["MAX_AUDIO_SEC"]], max_target_tokens=CFG["MAX_TARGET_TOKENS"], seed=CFG["SEED"],
    architecture=dict(train_enc=CFG["TRAIN_ENC_LAYERS"], train_dec=CFG["TRAIN_DEC_LAYERS"], enc_ffn_width=ENC_FFN_WIDTH, dec_ffn_width=DEC_FFN_WIDTH, enc_ln_trainable=True,
                      init="function-preserving widening: new fc1 ~ N(0, std_pretrained), new fc2 columns = 0"),
    specaugment=CFG["SPECAUGMENT"],
    optim=dict(lr=CFG["BASE_LR"], warmup=CFG["WARMUP_STEPS"], min_lr_ratio=CFG["MIN_LR_RATIO"], wd=CFG["WEIGHT_DECAY"], betas=list(CFG["BETAS"]), clip=CFG["GRAD_CLIP"],
               epochs=CFG["NUM_EPOCHS"], effective_bs=EFFECTIVE_BS, total_steps=TOTAL_STEPS),
    monitoring=dict(val_subset_per_lang=CFG["VAL_SUBSET_PER_LANG"], eval_steps=EVAL_STEPS, ckpt_topk=CFG["CKPT_TOPK"], early_stop_patience=CFG["EARLY_STOP_PATIENCE"], max_new_tokens=CFG["MAX_NEW_TOKENS"]),
    data=dict(n_train=N_TRAIN, train_per_lang=CFG["TRAIN_PER_LANG"], train_ids=_ids_md5(train_pool.ids), val_ids=_ids_md5(val_pool.ids), test_ids=_ids_md5(test_pool.ids)),
    recipe=CFG["RECIPE"], libs=dict(transformers=transformers.__version__, torch=torch.__version__, datasets=datasets.__version__))
EXPERIMENT_HASH = hashlib.sha256(json.dumps(FINGERPRINT, sort_keys=True).encode("utf-8")).hexdigest()[:12]

EXPERIMENT_CONFIG = dict(
    model=f"{CFG['MODEL_ID']} @ {MODEL_REVISION}", model_name=MODEL_NAME, dataset=CFG["DATASET_ID"], languages=[LANG_NAME[l] for l in LANGS],
    unique_clean_train_utterances=int(st_av.n.sum()), train_instances=N_TRAIN, train_instances_per_language={LANG_NAME[l]: int(st_tr.loc[l, "n"]) for l in LANGS},
    unique_train_utterances_used=int(st_tr.unique.sum()), validation_examples=len(val_pool), test_examples=len(test_pool), seed=CFG["SEED"], epochs=CFG["NUM_EPOCHS"],
    trainable_encoder_layers=[i + 1 for i in CFG["TRAIN_ENC_LAYERS"]], trainable_decoder_layers=[i + 1 for i in CFG["TRAIN_DEC_LAYERS"]], decoder_ffn_width=DEC_FFN_WIDTH, encoder_ffn_width=ENC_FFN_WIDTH, original_ffn_width=H0,
    specaugment=CFG["SPECAUGMENT"], optimizer=f"AdamW(betas={tuple(CFG['BETAS'])}, weight_decay={CFG['WEIGHT_DECAY']})", learning_rate=CFG["BASE_LR"],
    warmup_steps=WARMUP_STEPS, total_optimizer_steps=TOTAL_STEPS, micro_batch=MICRO_BS, gradient_accumulation=ACCUM, effective_batch=EFFECTIVE_BS,
    trainable_parameters=PARAM_COUNTS["model"]["trainable"], total_parameters=PARAM_COUNTS["model"]["total"],
    gpu=f"{GPU_NAME} ({GPU_VRAM_GB:.1f} GiB)", experiment_hash=EXPERIMENT_HASH, recipe=CFG["RECIPE"], leakage_check=LEAKAGE_REPORT)

def print_final_configuration():
    print("=" * 50); print("FINAL EXPERIMENT CONFIGURATION"); print("=" * 50)
    print(f"\nModel:\n  {MODEL_NAME}  (single dense Whisper-small, function-preserving scaling)")
    print("\nLanguages:\n  " + "\n  ".join(LANG_NAME[l] for l in LANGS))
    print(f"\nTraining instances:\n  {N_TRAIN:,} ({CFG['NUM_EPOCHS']} epochs x {CFG['TRAIN_PER_LANG']} per language x {len(LANGS)}; {int(st_tr.unique.sum()):,} unique utterances of {int(st_av.n.sum()):,} available)")
    print(f"\nTrainable layers:\n  encoder {[i + 1 for i in CFG['TRAIN_ENC_LAYERS']]} + final LayerNorm, decoder {[i + 1 for i in CFG['TRAIN_DEC_LAYERS']]} + final LayerNorm")
    print(f"\nFFN widths (trainable layers):\n  encoder {D_MODEL} -> {ENC_FFN_WIDTH} -> {D_MODEL} | decoder {D_MODEL} -> {DEC_FFN_WIDTH} -> {D_MODEL}   (original {D_MODEL} -> {H0} -> {D_MODEL}; function-preserving widening)")
    print(f"\nParameters:\n  trainable {PARAM_COUNTS['model']['trainable']:,} | total {PARAM_COUNTS['model']['total']:,}")
    print(f"\nSpecAugment:\n  {CFG['SPECAUGMENT']}")
    print(f"\nEpochs / steps:\n  {CFG['NUM_EPOCHS']} epochs, {TOTAL_STEPS} optimiser steps, warm-up {WARMUP_STEPS}, cosine to {CFG['MIN_LR_RATIO']:.0%}")
    print(f"\nEffective batch:\n  {EFFECTIVE_BS} = micro-batch {MICRO_BS} (from the memory estimate) x accumulation {ACCUM}")
    print(f"\nLearning rate:\n  {CFG['BASE_LR']}  (AdamW, grad-clip {CFG['GRAD_CLIP']}, FP16 autocast + GradScaler)")
    print(f"\nSelection:\n  validation-subset WER every {CFG['EVAL_EVERY_STEPS']} steps -> best {CFG['CKPT_TOPK']} -> full validation set picks the final checkpoint; early-stop patience {CFG['EARLY_STOP_PATIENCE']} evaluations")
    print("\nTraining runs:\n  EXACTLY 1 (single dense model)"); print("=" * 50)
print_final_configuration()
with open(f"{OUT_DIR}/experiment_config.json", "w") as f: json.dump(EXPERIMENT_CONFIG, f, indent=1, default=str)
print(f"\nGPU                  : {GPU_NAME}\nVRAM                 : {GPU_VRAM_GB:.1f} GiB\nmicro-batch size     : {MICRO_BS}\ngrad accumulation    : {ACCUM}\neffective batch size : {EFFECTIVE_BS}")
print(f"training instances   : {N_TRAIN}\ntotal optimizer steps: {TOTAL_STEPS}  ({STEPS_PER_EPOCH:.0f} per epoch; the last {DROPPED_TRAIN} instances of the pool do not fill an accumulation window and are not used)")
print(f"warm-up steps        : {WARMUP_STEPS} ({100*WARMUP_STEPS/TOTAL_STEPS:.0f} % of the run)  -> cosine to {CFG['MIN_LR_RATIO']:.0%} of peak")
print(f"validation points    : steps {EVAL_STEPS}")
print(f"experiment hash      : {EXPERIMENT_HASH}  (checkpoints / predictions are re-used only when this matches)")

## Monitoring, evaluation helpers and checkpointing
Cells 35-36 are defined *before* the training run (Cells 34 / 36B) because the training loop calls them.
Cell 40's text normaliser is also defined here for the same reason; Cell 40 itself demonstrates it.

In [ ]:
def normalize_eval(t):
    """NFC -> punctuation/symbol -> space (Unicode categories P*, S*) -> lower-case Latin letters only (Unicode name contains LATIN) -> collapse whitespace.
    Combining marks (Mn/Mc), ZWJ/ZWNJ and all native-script letters/digits are preserved; nothing is transliterated."""
    t = unicodedata.normalize("NFC", t if isinstance(t, str) else "")
    t = "".join(" " if unicodedata.category(ch)[0] in ("P", "S") else (ch.lower() if is_latin_letter(ch) else ch) for ch in t)
    return re.sub(r"\s+", " ", t).strip()

def utt_stats(ref, hyp):
    r, h = normalize_eval(ref), normalize_eval(hyp)
    rw, hw = r.split(), h.split()
    return dict(ref_n=r, hyp_n=h, w_len=len(rw), w_err=Levenshtein.distance(rw, hw), c_len=len(r), c_err=Levenshtein.distance(r, h))

def add_scores(df):
    st = pd.DataFrame([utt_stats(a, b) for a, b in zip(df.ref, df.hyp)], index=df.index)
    df = pd.concat([df, st], axis=1)
    df["wer"] = np.where(df.w_len > 0, df.w_err / df.w_len.clip(lower=1), np.nan)
    df["cer"] = np.where(df.c_len > 0, df.c_err / df.c_len.clip(lower=1), np.nan)
    return df

def corpus_rates(df):
    d = df[df.w_len > 0]
    return dict(wer=d.w_err.sum() / max(1, d.w_len.sum()), cer=d.c_err.sum() / max(1, d.c_len.sum()), n=len(d))

def cut_at_eos(row):
    """(tokens before the first EOS, finished_by_eos). Robust to whatever pad id generate() uses after a sequence ends."""
    hit = (row == EOS_ID).nonzero()
    return (row[: int(hit[0])], True) if len(hit) else (row, False)

def make_lang_batches(pool, batch_size):
    by = defaultdict(list)
    for i, l in enumerate(pool.lang): by[l].append(i)
    batches = []
    for l in LANGS:
        idx = sorted(by.get(l, []), key=lambda i: -pool.dur[i])
        batches += [idx[j:j + batch_size] for j in range(0, len(idx), batch_size)]
    return batches

TEST_UNLOCKED = False
def guard_test(pool):
    assert TEST_UNLOCKED or getattr(pool, "hf", None) is not test_ds, "the FLEURS test set must not be decoded or scored before the final evaluation (Cell 38)"

@torch.no_grad()
def transcribe_pool(model, pool, batch_size=None, max_new_tokens=None, desc="decode", progress=True):
    """Greedy decoding (num_beams=1, no sampling), task=transcribe, forced language, no timestamps."""
    guard_test(pool); model.eval()
    bs, mnt = batch_size or CFG["EVAL_BATCH"], max_new_tokens or CFG["MAX_NEW_TOKENS"]
    loader = torch.utils.data.DataLoader(pool, batch_sampler=make_lang_batches(pool, bs), num_workers=CFG["NUM_WORKERS"], collate_fn=eval_collate)
    rows = []
    for b in tqdm(loader, desc=desc, leave=False, disable=not progress):
        feats = b["input_features"].to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16):     # NB: no attention_mask is passed on purpose - Whisper input_features are always zero-padded to 30 s (the pre-training contract), the encoder takes no mask,
                                                              # and every row of a batch shares the same decoder prompt (same language). transformers' "attention mask is not set" message is generic (pad == eos) and cosmetic here.
            seq = model.generate(input_features=feats, language=b["langs"][0], task="transcribe", return_timestamps=False,
                                 max_new_tokens=mnt, do_sample=False, num_beams=1)
        seq = seq.detach().cpu()
        cut = [cut_at_eos(seq[k]) for k in range(seq.shape[0])]
        hyps = tokenizer.batch_decode([c.tolist() for c, _ in cut], skip_special_tokens=True)       # plain python lists: a list of tensors raises in recent transformers
        for k, (c, fin) in enumerate(cut):
            rows.append(dict(id=b["ids"][k], dataset="fleurs", lang=b["langs"][k], audio_sec=round(b["audio_sec"][k], 3), ref=b["refs"][k],
                             hyp=clean_transcript(hyps[k]), finished_by_eos=fin, n_gen_tokens=int((c < EOS_ID).sum()) + int(fin)))
    return pd.DataFrame(rows)

@torch.no_grad()
def teacher_forced_loss(model, pool, batch_size=8):
    guard_test(pool); model.eval()
    loader = torch.utils.data.DataLoader(pool, batch_size=batch_size, shuffle=False, num_workers=CFG["NUM_WORKERS"], collate_fn=collator)
    tot, n = 0.0, 0
    for b in loader:
        with torch.autocast("cuda", dtype=torch.float16):
            out = run_forward(model, b)
        lab = b["labels"].to(DEVICE)
        tot += asr_loss(out.logits, lab, reduction="sum").item(); n += int((lab != -100).sum())
    return tot / max(1, n)

_rs = np.random.RandomState(CFG["SEED"])
_sel = []
for l in LANGS:
    ix = np.where(tables["validation"].lang.values == l)[0]
    _sel += sorted(_rs.choice(ix, size=min(CFG["VAL_SUBSET_PER_LANG"], len(ix)), replace=False).tolist())
val_subset_pool = FleursPool(tables["validation"].iloc[_sel].reset_index(drop=True), val_ds)
print("fixed validation subset:", len(val_subset_pool), "utterances (", CFG["VAL_SUBSET_PER_LANG"], "per language )")

def evaluate_subset(model, pool=None):
    pool = pool or val_subset_pool
    vloss = teacher_forced_loss(model, pool)
    df = add_scores(transcribe_pool(model, pool, desc="val-subset", progress=False))
    r = corpus_rates(df)
    per_lang = {l: corpus_rates(df[df.lang == l])["wer"] for l in LANGS}
    return dict(val_loss=vloss, val_wer=r["wer"], val_cer=r["cer"], per_lang_wer=per_lang)

In [ ]:
def trainable_state(model):
    return {n: p.detach().cpu().contiguous() for n, p in model.named_parameters() if p.requires_grad}

def save_trainable_ckpt(model, ckpt_dir, meta):
    os.makedirs(ckpt_dir, exist_ok=True)
    save_file(trainable_state(model), f"{ckpt_dir}/trainable.safetensors")
    with open(f"{ckpt_dir}/config.json", "w") as f: json.dump(meta, f, indent=1)

def load_trainable_ckpt(model, ckpt_dir):
    sd = load_file(f"{ckpt_dir}/trainable.safetensors")
    expected = {n for n, p in model.named_parameters() if p.requires_grad}
    assert set(sd) == expected, f"checkpoint/trainable-parameter mismatch: {len(set(sd) ^ expected)} names differ"
    res = model.load_state_dict(sd, strict=False)
    assert not res.unexpected_keys, res.unexpected_keys
    return model

def ckpt_path(tag, step): return f"{OUT_DIR}/ckpt_{tag}/step{step}"

def ckpt_meta(tag, step, val):
    return dict(experiment_hash=EXPERIMENT_HASH, model_name=MODEL_NAME, arch="dense_widened", enc_ffn_width=ENC_FFN_WIDTH, dec_ffn_width=DEC_FFN_WIDTH, widened_enc_layers=CFG["TRAIN_ENC_LAYERS"], widened_dec_layers=CFG["TRAIN_DEC_LAYERS"], hidden_size=D_MODEL, tag=tag, step=step,
                val_subset_wer=val["val_wer"], model_id=CFG["MODEL_ID"], model_revision=MODEL_REVISION, train_enc_layers=CFG["TRAIN_ENC_LAYERS"], train_dec_layers=CFG["TRAIN_DEC_LAYERS"],
                trainable_parameters=PARAM_COUNTS["model"]["trainable"], total_parameters=PARAM_COUNTS["model"]["total"], transformers=transformers.__version__)

In [ ]:
def make_scaler():
    try: return torch.amp.GradScaler("cuda")
    except Exception: return torch.cuda.amp.GradScaler()

def make_train_loader(start=0):
    g = torch.Generator(); g.manual_seed(CFG["SEED"])
    ds = train_pool if start == 0 else torch.utils.data.Subset(train_pool, range(start, len(train_pool)))
    return torch.utils.data.DataLoader(ds, batch_size=MICRO_BS, shuffle=False, drop_last=True, generator=g,
                                       num_workers=CFG["NUM_WORKERS"], collate_fn=collator, prefetch_factor=8 if CFG["NUM_WORKERS"] > 0 else None)

def load_run(tag):
    with open(f"{OUT_DIR}/run_{tag}.json") as f: run = json.load(f)
    run["log"] = pd.read_csv(f"{OUT_DIR}/trainlog_{tag}.csv"); run["val"] = pd.read_csv(f"{OUT_DIR}/vallog_{tag}.csv")
    return run

def slice_batch(b, lo, hi):
    """Rows lo:hi of a collated batch; the decoder length is trimmed to the longest sequence of the slice."""
    T = int(b["lens"][lo:hi].max()) - 1
    return dict(input_features=b["input_features"][lo:hi], decoder_input_ids=b["decoder_input_ids"][lo:hi, :T],
                labels=b["labels"][lo:hi, :T], valid=b["valid"][lo:hi, :T], lens=b["lens"][lo:hi])

def run_window(model, window, chunk, scaler):
    """Forward + backward for ONE optimiser step = ACCUM micro-batches, each processed in pieces of <= `chunk` examples.
    Objective = cross-entropy summed over every target token of the WHOLE effective batch / number of those tokens (a global token mean,
    independent of how the batch is split into micro-batches / OOM-fallback pieces)."""
    n_tok = max(1, sum(int((b["labels"] != -100).sum()) for b in window))
    st = dict(loss=0., tok=n_tok, ex=0)
    for b in window:
        B = b["input_features"].shape[0]
        for lo in range(0, B, chunk):
            hi = min(B, lo + chunk)
            sb = slice_batch(b, lo, hi)
            with torch.autocast("cuda", dtype=torch.float16):
                out = run_forward(model, sb)
            labels = sb["labels"].to(DEVICE, non_blocking=True)
            loss = asr_loss(out.logits, labels, reduction="sum") / n_tok; del out
            scaler.scale(loss).backward()
            st["loss"] += loss.item(); st["ex"] += hi - lo
    return st

def train_model(model, tag):
    ckpt_dir = f"{OUT_DIR}/ckpt_{tag}"
    rp = f"{OUT_DIR}/run_{tag}.json"
    if os.path.exists(rp):
        with open(rp) as f: old = json.load(f)
        cands_ok = bool(old.get("candidates")) and all(os.path.exists(f"{ckpt_path(tag, c['step'])}/trainable.safetensors") for c in old["candidates"])
        if old.get("experiment_hash") == EXPERIMENT_HASH and cands_ok:
            _log(f"[{MODEL_NAME}] completed run with the SAME experiment hash ({EXPERIMENT_HASH}) found on disk - skipping training"); return load_run(tag)
        stale = f"{OUT_DIR}/stale/{old.get('experiment_hash', 'unknown')}_{tag}"; os.makedirs(stale, exist_ok=True)
        for p_ in (rp, f"{OUT_DIR}/trainlog_{tag}.csv", f"{OUT_DIR}/vallog_{tag}.csv", ckpt_dir):
            if os.path.exists(p_): shutil.move(p_, stale)
        _log(f"[{MODEL_NAME}] found a run from a DIFFERENT / incomplete experiment ({old.get('experiment_hash', 'unknown')}); moved it to {stale} and training afresh")
    os.makedirs(SCRATCH, exist_ok=True); resume_path = f"{SCRATCH}/resume_{tag}.pt"; state = None
    if os.path.exists(resume_path):
        try:
            state = torch.load(resume_path, map_location="cpu", weights_only=False)
            ok_ = (state.get("experiment_hash") == EXPERIMENT_HASH and state.get("arch") == "dense_widened" and state.get("fingerprint") == FINGERPRINT
                   and all(os.path.exists(f"{ckpt_path(tag, s_)}/trainable.safetensors") for _, s_ in state["cands"]))
        except Exception as e_:
            ok_ = False; print(f"[{MODEL_NAME}] resume snapshot unreadable ({type(e_).__name__}: {str(e_)[:80]})")
        if not ok_:
            state = None; os.remove(resume_path); _log(f"[{MODEL_NAME}] discarded a resume snapshot that does not belong to this experiment ({EXPERIMENT_HASH})")
    if state is None: shutil.rmtree(ckpt_dir, ignore_errors=True)
    print("\n" + "=" * 16 + " EXPERIMENT " + "=" * 16 + f"\n{MODEL_NAME}\n" + "=" * 44)
    c_ = count_params(model)
    print(f"total parameters {c_['total']:,} | trainable {c_['trainable']:,} | frozen {c_['frozen']:,}")
    assert c_["trainable"] == PARAM_COUNTS["model"]["trainable"], "trainable-parameter count differs from the verified Cell 27 count"
    random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"]); torch.manual_seed(CFG["SEED"]); torch.cuda.manual_seed_all(CFG["SEED"])
    model.to(DEVICE); set_train_mode(model)                                       # frozen modules stay in eval(); the WhisperModel stays in train mode (SpecAugment)
    model.generation_config.max_length = None
    params = [p for p in model.parameters() if p.requires_grad]
    opt, scaler = build_optimizer(model), make_scaler()
    sched = make_scheduler(opt, TOTAL_STEPS)
    log_rows, val_rows, cands, best = [], [], [], dict(val_wer=float("inf"), step=-1)
    fresh = lambda: dict(loss=0., tok=0, ex=0, gn=0., steps=0)
    win = fresh(); t_win = time.time()
    t0 = time.time(); eval_secs = 0.0; step = 0; chunk = MICRO_BS; oom_events = skipped = bad_loss = bad_skip = no_improve = ex_done = resumed_from = 0; v = None; early_stopped = False
    torch.cuda.reset_peak_memory_stats(); opt.zero_grad(set_to_none=True)
    if state is not None:
        res_ = model.load_state_dict(state["model"], strict=False); assert not res_.unexpected_keys, res_.unexpected_keys
        opt.load_state_dict(state["opt"]); sched.load_state_dict(state["sched"]); scaler.load_state_dict(state["scaler"])
        step, cands, best, no_improve, v = state["step"], state["cands"], state["best"], state["no_improve"], state["v"]
        log_rows, val_rows, eval_secs, oom_events, skipped, chunk, ex_done = (state[k] for k in ("log_rows", "val_rows", "eval_secs", "oom_events", "skipped", "chunk", "ex_done"))
        t0 = time.time() - state["wall"]; resumed_from = step
        random.seed(CFG["SEED"] + step); np.random.seed(CFG["SEED"] + step)
        _log(f"[{MODEL_NAME}] RESUMED from the snapshot at step {step}/{TOTAL_STEPS} (optimizer, scheduler, scaler, best candidates restored)"); del state
    loader = make_train_loader(step * EFFECTIVE_BS); it = iter(loader)
    _log(f"[{MODEL_NAME}] training: {TOTAL_STEPS} steps ({CFG['NUM_EPOCHS']} epochs), trainable {sum(p.numel() for p in params)/1e6:.1f} M params, micro-batch {MICRO_BS} x accum {ACCUM}, SpecAugment={CFG['SPECAUGMENT']['enabled']}")
    while step < TOTAL_STEPS:
        window = [next(it) for _ in range(ACCUM)]
        while True:
            oom = False
            try: st = run_window(model, window, chunk, scaler)
            except torch.cuda.OutOfMemoryError: oom = True
            if not oom: break
            opt.zero_grad(set_to_none=True); gc.collect(); torch.cuda.empty_cache()   # (outside the except block so the traceback frees its tensors)
            if chunk == 1: raise RuntimeError("CUDA out of memory even with a single example per forward pass")
            chunk = max(1, chunk // 2); oom_events += 1
            print(f"[{MODEL_NAME}] CUDA OOM -> repeating this optimiser step with <= {chunk} example(s) per forward pass", flush=True)
        if not math.isfinite(st["loss"]):
            bad_loss += 1
            if bad_loss >= CFG["MAX_CONSEC_NONFINITE"]:
                raise FloatingPointError(f"[{MODEL_NAME}] non-finite training loss in {bad_loss} consecutive optimiser steps (step {step + 1}); last window: {st}")
        else: bad_loss = 0
        for k in st: win[k] += st[k]
        ex_done += st["ex"]
        scaler.unscale_(opt)
        gn = torch.nn.utils.clip_grad_norm_(params, CFG["GRAD_CLIP"]); win["gn"] += float(gn) if torch.isfinite(gn) else 0.0; win["steps"] += 1
        lr_now = opt.param_groups[0]["lr"]
        scale_before = scaler.get_scale()
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
        if scaler.get_scale() < scale_before:                                      # fp16 overflow (non-finite gradients) -> optimiser step skipped -> LR schedule does NOT advance
            skipped += 1; bad_skip += 1
            if bad_skip >= CFG["MAX_CONSEC_SKIPPED"]: raise FloatingPointError(f"[{MODEL_NAME}] {bad_skip} consecutive optimiser steps skipped because of non-finite gradients (step {step})")
        else: sched.step(); bad_skip = 0
        if step % CFG["LOG_EVERY_STEPS"] == 0 or step == TOTAL_STEPS:
            dt = time.time() - t_win; n = max(1, win["steps"])
            row = dict(step=step, epoch=step / STEPS_PER_EPOCH, loss=win["loss"] / n, asr_loss=win["loss"] / n, grad_norm=win["gn"] / n, lr=lr_now, tokens_per_sec=win["tok"] / dt, examples_per_sec=win["ex"] / dt,
                       elapsed_min=(time.time() - t0) / 60, gpu_mem_gb=torch.cuda.max_memory_allocated() / 1024**3, grad_scale=float(scaler.get_scale()), skipped_steps=skipped, chunk=chunk)
            log_rows.append(row)
            print(f"[{MODEL_NAME}] step {step:4d}/{TOTAL_STEPS} (epoch {row['epoch']:.2f}) loss {row['loss']:.4f} lr {row['lr']:.2e} | {row['tokens_per_sec']:.0f} tok/s {row['examples_per_sec']:.1f} ex/s | "
                  f"{row['elapsed_min']:.1f} min | {row['gpu_mem_gb']:.1f} GiB", flush=True)
            win = fresh(); t_win = time.time(); torch.cuda.reset_peak_memory_stats()
        if step in EVAL_STEPS:
            te = time.time(); v = evaluate_subset(model); set_train_mode(model)
            eval_secs += time.time() - te
            val_rows.append(dict(step=step, epoch=step / STEPS_PER_EPOCH, val_loss=v["val_loss"], val_wer=v["val_wer"], val_cer=v["val_cer"], **{f"wer_{l}": v["per_lang_wer"][l] for l in LANGS}))
            print(f"[{MODEL_NAME}] >>> VAL step {step} (epoch {step / STEPS_PER_EPOCH:.2f}): loss {v['val_loss']:.4f} | WER {v['val_wer']:.4f} | CER {v['val_cer']:.4f}", flush=True)
            prev_best = min((c[0] for c in cands), default=float("inf"))               # best subset WER BEFORE this evaluation (the first evaluation establishes the first best)
            new_best = v["val_wer"] < prev_best                                        # genuine improvement of the best score -> resets early-stopping patience
            keep = len(cands) < CFG["CKPT_TOPK"] or v["val_wer"] < max(c[0] for c in cands)   # top-K retention is independent of the patience counter
            if keep:
                save_trainable_ckpt(model, ckpt_path(tag, step), ckpt_meta(tag, step, v)); cands.append((v["val_wer"], step))
                if len(cands) > CFG["CKPT_TOPK"]:
                    worst = max(cands); cands.remove(worst); shutil.rmtree(ckpt_path(tag, worst[1]), ignore_errors=True)
                print(f"[{MODEL_NAME}]     checkpoint kept (val-subset WER {v['val_wer']:.4f}); candidates now: {sorted((s, round(w, 4)) for w, s in cands)}", flush=True)
            no_improve = 0 if new_best else no_improve + 1                             # patience counts validation EVALUATIONS without a new best subset WER
            best = dict(val_wer=min(cands)[0], step=min(cands)[1])
            if step < TOTAL_STEPS:
                _ts = time.time(); _tmp = resume_path + ".tmp"
                torch.save(dict(experiment_hash=EXPERIMENT_HASH, arch="dense_widened", fingerprint=FINGERPRINT, step=step, epoch=step / STEPS_PER_EPOCH, model=trainable_state(model), opt=opt.state_dict(),
                                sched=sched.state_dict(), scaler=scaler.state_dict(), cands=cands, best=best, no_improve=no_improve, v=v, log_rows=log_rows, val_rows=val_rows, eval_secs=eval_secs,
                                oom_events=oom_events, skipped=skipped, chunk=chunk, ex_done=ex_done, wall=time.time() - t0), _tmp)
                os.replace(_tmp, resume_path); eval_secs += time.time() - _ts
            if no_improve >= CFG["EARLY_STOP_PATIENCE"] and step >= STEPS_PER_EPOCH and step < TOTAL_STEPS:
                early_stopped = True
                print(f"[{MODEL_NAME}] early stopping at step {step}: no new best validation-subset WER in {no_improve} evaluations (best {best['val_wer']:.4f} at step {best['step']})", flush=True); break
    wall = time.time() - t0
    save_trainable_ckpt(model, f"{OUT_DIR}/{TAG}_final", ckpt_meta(tag, step, v))
    run = dict(tag=tag, model_name=MODEL_NAME, steps=step, early_stopped=early_stopped, best_step=best["step"], best_val_wer=best["val_wer"], train_seconds=wall - eval_secs, eval_seconds=eval_secs,
               wall_seconds=wall, avg_step_seconds=(wall - eval_secs) / max(1, step), peak_gpu_mem_gb=max(r["gpu_mem_gb"] for r in log_rows),
               trainable_params=sum(p.numel() for p in params), micro_bs=MICRO_BS, accum=ACCUM, effective_bs=EFFECTIVE_BS, total_steps=TOTAL_STEPS,
               oom_events=oom_events, final_chunk=chunk, skipped_optimizer_steps=skipped, experiment_hash=EXPERIMENT_HASH,
               epochs_completed=step / STEPS_PER_EPOCH, optimizer_steps=step, examples_processed=ex_done, throughput_examples_per_sec=ex_done / max(1e-9, wall - eval_secs), resumed_from_step=resumed_from,
               candidates=[dict(step=s, val_subset_wer=w) for w, s in sorted(cands, key=lambda c: c[1])])
    log_df, val_df = pd.DataFrame(log_rows), pd.DataFrame(val_rows)
    log_df.to_csv(f"{OUT_DIR}/trainlog_{tag}.csv", index=False); val_df.to_csv(f"{OUT_DIR}/vallog_{tag}.csv", index=False)
    with open(f"{OUT_DIR}/run_{tag}.json", "w") as f: json.dump(run, f, indent=1)
    run["log"], run["val"] = log_df, val_df
    if os.path.exists(resume_path): os.remove(resume_path)
    model.cpu(); del opt, scaler, sched, loader, it; gc.collect(); torch.cuda.empty_cache()
    _log(f"[{MODEL_NAME}] done: best val-subset WER {best['val_wer']:.4f} at step {best['step']} | train {run['train_seconds']/60:.1f} min (+{eval_secs/60:.1f} min monitoring) | peak GPU {run['peak_gpu_mem_gb']:.1f} GB | OOM retries {oom_events} | skipped fp16 steps {skipped} | early stopped: {early_stopped}")
    return run

In [ ]:
set_train_mode(model)
enc_, dec_ = model.model.encoder, model.model.decoder
_trainable_layer = lambda L: all(p.requires_grad for p in L.parameters())
_frozen_layer = lambda L: not any(p.requires_grad for p in L.parameters())
tr_enc = [i for i in range(12) if _trainable_layer(enc_.layers[i])]; tr_dec = [i for i in range(12) if _trainable_layer(dec_.layers[i])]
fr_enc = [i for i in range(12) if _frozen_layer(enc_.layers[i])];    fr_dec = [i for i in range(12) if _frozen_layer(dec_.layers[i])]
_opt_probe = build_optimizer(model); _opt_ids = {id(p) for g in _opt_probe.param_groups for p in g["params"]}; del _opt_probe
pc_ = count_params(model)
_mod_names = [f"{n}/{type(m_).__name__}".lower() for n, m_ in model.named_modules()]
_per_split = {s: set(tables[s].lang) for s in ("train_available", "train", "validation", "test")}
_epoch_counts = [b_.groupby("lang").size().to_dict() for b_ in epoch_blocks]
_gpu_ok = torch.cuda.is_available() and GPU_VRAM_GB > 10
CHECKS = [
    ("GPU detected", f"{GPU_NAME}", _gpu_ok),
    ("VRAM detected", f"{GPU_VRAM_GB:.2f} GiB", GPU_VRAM_GB > 10),
    ("Whisper revision is the pinned commit", MODEL_REVISION[:12], MODEL_REVISION == CFG["MODEL_REVISION_PIN"]),
    ("encoder layers 7-12 trainable, 1-6 frozen", f"trainable {[i + 1 for i in tr_enc]}", tr_enc == list(range(6, 12)) and fr_enc == list(range(0, 6))),
    ("decoder layers 7-12 trainable, 1-6 frozen", f"trainable {[i + 1 for i in tr_dec]}", tr_dec == list(range(6, 12)) and fr_dec == list(range(0, 6))),
    ("encoder FFN = %d in layers 7-12 (3072 elsewhere)" % ENC_FFN_WIDTH, "", all(enc_.layers[i].fc1.out_features == (ENC_FFN_WIDTH if i >= 6 else H0) for i in range(12))),
    ("decoder FFN = %d in layers 7-12 (3072 elsewhere)" % DEC_FFN_WIDTH, "", all(dec_.layers[i].fc1.out_features == (DEC_FFN_WIDTH if i >= 6 else H0) for i in range(12))),
    ("embeddings, conv front-end, positional encodings frozen", "", not any(p.requires_grad for m_ in (dec_.embed_tokens, dec_.embed_positions, enc_.conv1, enc_.conv2, enc_.embed_positions) for p in m_.parameters())),
    ("output head tied to the token embedding and frozen", "", model.proj_out.weight.data_ptr() == dec_.embed_tokens.weight.data_ptr() and not model.proj_out.weight.requires_grad),
    ("parameter counts: instantiated model == architecture specification", f"total {pc_['total']:,} | trainable {pc_['trainable']:,} ({100 * pc_['trainable'] / pc_['total']:.1f} %) | frozen {pc_['frozen']:,}",
        pc_ == PARAM_COUNTS["model"] and pc_["trainable"] == EXPECTED_TRAINABLE_PARAMS and pc_["total"] == EXPECTED_TOTAL_PARAMS),
    ("function-preserving initialisation + FFN equivalence tests (Cell 26) and per-layer weight checks (Cell 27)", f"max logit err {EQUIV_RESULT['model_logit_max_abs_err']:.1e}", EQUIV_RESULT["passed"]),
    ("no MoE / router / expert modules exist", "", not any(re.search(r"moe|expert|router|gating", s_) for s_ in _mod_names)),
    ("frozen modules in eval(), trainable modules in train()", "", (not any(enc_.layers[i].training for i in fr_enc)) and all(enc_.layers[i].training for i in tr_enc) and (not any(dec_.layers[i].training for i in fr_dec))
        and all(dec_.layers[i].training for i in tr_dec) and not dec_.embed_tokens.training and not enc_.conv1.training and enc_.layer_norm.training and dec_.layer_norm.training and model.model.training),
    ("SpecAugment configured (active only in train mode; tested in Cell 26)", f"{CFG['SPECAUGMENT']['time_prob']:.0%} time / {CFG['SPECAUGMENT']['feature_prob']:.0%} mel", bool(model.config.apply_spec_augment) == CFG["SPECAUGMENT"]["enabled"] and EQUIV_RESULT["passed"]),
    ("forward/backward are not wrapped in torch.no_grad (gradients flow through the frozen layers into the trainable ones)", "", "no_grad" not in inspect.getsource(run_forward) + inspect.getsource(run_window)),
    ("optimizer contains ONLY trainable parameters (and all of them)", f"{len(_opt_ids)} tensors", _opt_ids == {id(p) for p in model.parameters() if p.requires_grad}),
    ("all 8 languages present in train / validation / test", ", ".join(LANGS), all(_per_split[s] == set(LANGS) for s in _per_split)),
    ("language balance: every language contributes exactly %d instances per epoch" % CFG["TRAIN_PER_LANG"], "", all(c_ == {l: CFG["TRAIN_PER_LANG"] for l in LANGS} for c_ in _epoch_counts)),
    ("train / validation / test pairwise disjoint (sentence ids + transcripts, Cell 15c)", "", all(r_["shared_sentence_ids"] == 0 for r_ in LEAKAGE_REPORT.values())),
    ("test set not decoded or scored so far (guard closed until Cell 38)", "", not TEST_UNLOCKED),
    ("audio duration filter: every kept clip is %.0f-%.0f s" % (CFG["MIN_AUDIO_SEC"], CFG["MAX_AUDIO_SEC"]), "", bool(meta_final.dur.between(CFG["MIN_AUDIO_SEC"], CFG["MAX_AUDIO_SEC"]).all())),
    ("token-length filter: prefix + text + EOS <= %d for every kept example" % CFG["MAX_TARGET_TOKENS"], "", int(meta_final.n_target.max()) <= CFG["MAX_TARGET_TOKENS"] and max(len(t_) for t_ in train_pool.text_ids) + CFG["N_PREFIX"] + 1 <= CFG["MAX_TARGET_TOKENS"]),
    ("loss alignment verified on real collator output (Cell 29); EOS kept as a target, prefix and padding excluded", "", LOSS_ALIGNMENT_OK),
    ("language tokens verified for all 8 languages (Cell 29)", "", len(set(LANG_TOKEN_IDS.values())) == len(LANGS) == 8),
    ("micro-batch chosen from the worst-case memory estimate", f"micro-batch {MICRO_BS}, worst-case estimate {sum(estimate_memory_gb(MICRO_BS)):.1f} GiB of {GPU_VRAM_GB:.1f} GiB (budget {MEM_BUDGET_GB:.1f} GiB)", sum(estimate_memory_gb(MICRO_BS)) <= MEM_BUDGET_GB or MICRO_BS == 1),
    ("gradient accumulation x micro-batch = effective batch = %d" % CFG["TARGET_EFFECTIVE_BATCH"], f"{MICRO_BS} x {ACCUM} = {EFFECTIVE_BS}", MICRO_BS * ACCUM == EFFECTIVE_BS == CFG["TARGET_EFFECTIVE_BATCH"]),
    ("checkpoint selection on validation (subset for monitoring, full validation set for the final choice)", f"{CFG['VAL_SUBSET_PER_LANG']} utt/language, top-{CFG['CKPT_TOPK']}", val_subset_pool.hf is val_ds and CFG["CKPT_TOPK"] >= 1),
    ("early-stopping patience counts validation evaluations", f"{CFG['EARLY_STOP_PATIENCE']} evaluations, only after epoch 1", CFG["EARLY_STOP_PATIENCE"] >= 1 and len(EVAL_STEPS) >= CFG["EARLY_STOP_PATIENCE"] + 1),
    ("test evaluation disabled until the final stage", "", not TEST_UNLOCKED and test_pool.hf is test_ds),
]
print("PRE-FLIGHT CHECKLIST")
for desc_, info_, ok_ in CHECKS: print(f"  [{'x' if ok_ else ' '}] {desc_}" + (f"  ->  {info_}" if info_ else ""))
PREFLIGHT_OK = all(ok_ for _, _, ok_ in CHECKS)
if not PREFLIGHT_OK: raise AssertionError("PRE-FLIGHT FAILED for: " + "; ".join(d_ for d_, _, ok_ in CHECKS if not ok_))
print(f"\nALL {len(CHECKS)} PRE-FLIGHT CHECKS PASSED - the training cell may start. Detected GPU {GPU_NAME} ({GPU_VRAM_GB:.1f} GiB) | micro-batch {MICRO_BS} x accumulation {ACCUM} = effective batch {EFFECTIVE_BS}")

In [ ]:
RUNS = {}
assert PREFLIGHT_OK, "pre-flight checklist (Cell 35b) did not pass"
RUNS[TAG] = train_model(model, TAG)
del model; gc.collect(); torch.cuda.empty_cache()
_r = RUNS[TAG]
print(f"{MODEL_NAME}: training time {_r['train_seconds']/60:.1f} min (baseline Dense-198M: {CFG['PREVIOUS_DENSE_198M']['train_minutes']:.0f} min) | epochs completed {_r['epochs_completed']:.2f} | optimiser steps {_r['optimizer_steps']} | "
      f"examples processed {_r['examples_processed']:,} ({_r['throughput_examples_per_sec']:.1f} ex/s) | peak GPU memory {_r['peak_gpu_mem_gb']:.2f} GiB | OOM retries {_r['oom_events']}")

## Final evaluation
The selected checkpoint decodes the **complete** FLEURS validation and test sets with greedy decoding (`task=transcribe`, forced language, no timestamps, `max_new_tokens=440`). The test set is used for decoding / scoring only from here on, after the checkpoint has been chosen on validation; no decision (checkpointing, hyper-parameters, thresholds) depends on it. Test results refer to the filtered, cap-compatible subset produced by the fixed preprocessing pipeline.

In [ ]:
EVAL_SETS = {"validation": val_pool, "test": test_pool}
LOAD_SECONDS, DECODE_SECONDS = {}, {}

def pred_path(split, step): return f"{OUT_DIR}/pred_{TAG}_{split}_{EXPERIMENT_HASH}_step{step}.jsonl"

def load_model_at(step):
    d = ckpt_path(TAG, step)
    with open(f"{d}/config.json") as f: cfg_ = json.load(f)
    assert cfg_.get("experiment_hash") == EXPERIMENT_HASH, f"{d} belongs to another experiment ({cfg_.get('experiment_hash')} != {EXPERIMENT_HASH})"
    assert (cfg_.get("arch"), cfg_.get("enc_ffn_width"), cfg_.get("dec_ffn_width"), cfg_.get("trainable_parameters")) == ("dense_widened", ENC_FFN_WIDTH, DEC_FFN_WIDTH, PARAM_COUNTS["model"]["trainable"]), f"{d}: architecture fingerprint mismatch"
    t = time.time()
    m = build_model(); load_trainable_ckpt(m, d); m = m.to(DEVICE).eval()
    assert count_params(m) == PARAM_COUNTS["model"], "parameter fingerprint of the restored model differs from the verified model"
    LOAD_SECONDS[step] = time.time() - t
    return m

def decode_split_cached(split, step, model=None):
    """Greedy-decode a whole split with the checkpoint `step`; cached on disk (key = experiment hash + split + step). Returns (df, model-or-None)."""
    path = pred_path(split, step)
    if os.path.exists(path):
        _log(f"[{split}/step{step}] cached predictions"); return pd.read_json(path, lines=True, dtype=False), model
    if model is None: model = load_model_at(step)
    t = time.time(); df = transcribe_pool(model, EVAL_SETS[split], desc=f"{split}/step{step}")
    df.to_json(path, orient="records", lines=True, force_ascii=False); DECODE_SECONDS[(split, step)] = time.time() - t
    _log(f"[{split}/step{step}] decoded {len(df)} utterances in {(time.time()-t)/60:.1f} min")
    return df, model

cands_ = sorted(RUNS[TAG]["candidates"], key=lambda c: c["step"]); rows_ = []
for c in cands_:
    if len(cands_) == 1: wf = None
    else:
        df_, mdl_ = decode_split_cached("validation", c["step"])
        assert len(df_) == len(val_pool)
        wf = corpus_rates(add_scores(df_))["wer"]
        if mdl_ is not None: del mdl_; gc.collect(); torch.cuda.empty_cache()
    rows_.append(dict(step=c["step"], epoch=round(c["step"] / STEPS_PER_EPOCH, 2), val_subset_wer=c["val_subset_wer"], full_val_wer=wf))
SELECTED_STEP = cands_[0]["step"] if len(cands_) == 1 else min(rows_, key=lambda r: (r["full_val_wer"], r["step"]))["step"]
SELECTION = dict(candidates=rows_, selected_step=SELECTED_STEP, early_stopped=RUNS[TAG].get("early_stopped"), last_step=RUNS[TAG]["steps"])
print(f"checkpoint candidates {rows_} -> selected step {SELECTED_STEP} (epoch {SELECTED_STEP / STEPS_PER_EPOCH:.2f} of {CFG['NUM_EPOCHS']}); training ended at step {RUNS[TAG]['steps']}/{TOTAL_STEPS}")
_m = load_model_at(SELECTED_STEP)
_bp = dict(base_cpu.named_parameters())
changed = sum(not torch.equal(p.cpu(), _bp[n]) for n, p in _m.named_parameters() if n in _bp and p.requires_grad and p.shape == _bp[n].shape)
print("restored model in eval mode:", not _m.training, "| trainable tensors that differ from pretrained:", changed, "| load time %.1fs" % LOAD_SECONDS[SELECTED_STEP])
del _m; gc.collect(); torch.cuda.empty_cache()
def load_best_model(): return load_model_at(SELECTED_STEP)

In [ ]:
TEST_UNLOCKED = True
PRED = {}; model = None
for split in EVAL_SETS:
    df, model = decode_split_cached(split, SELECTED_STEP, model)
    assert len(df) == len(EVAL_SETS[split]) and df.id.nunique() == len(df)
    PRED[split] = df
if model is not None: del model; gc.collect(); torch.cuda.empty_cache()
display(PRED["test"].head(3)[["id", "lang", "audio_sec", "ref", "hyp", "finished_by_eos"]])

In [ ]:
# CELL 39 - WER / CER per language, pooled (micro) and macro
SCORED = {s: add_scores(d) for s, d in PRED.items()}

def rate_table(split):
    d = SCORED[split]; rows = []
    for l in LANGS:
        rr = corpus_rates(d[d.lang == l]); rows.append({"Language": LANG_NAME[l], "WER %": 100 * rr["wer"], "CER %": 100 * rr["cer"], "n": rr["n"]})
    mic = corpus_rates(d); per = [corpus_rates(d[d.lang == l]) for l in LANGS]
    rows.append({"Language": "Pooled (micro: all utterances)", "WER %": 100 * mic["wer"], "CER %": 100 * mic["cer"], "n": mic["n"]})
    rows.append({"Language": "Macro-average (mean of the 8 languages)", "WER %": 100 * np.mean([p["wer"] for p in per]), "CER %": 100 * np.mean([p["cer"] for p in per]), "n": mic["n"]})
    return pd.DataFrame(rows).set_index("Language")

POOLED, MACRO = "Pooled (micro: all utterances)", "Macro-average (mean of the 8 languages)"
def row_name(l): return LANG_NAME[l] if l in LANGS else POOLED
RATE_TABLES = {s: rate_table(s) for s in EVAL_SETS}
for s in EVAL_SETS:
    print(f"\n=== {MODEL_NAME} - FLEURS {s.upper()} ({'development / checkpoint selection' if s == 'validation' else 'FINAL held-out'}) ===")
    display(RATE_TABLES[s].round(2))
print("Pooled (micro) = total errors / total reference words (or characters) over all utterances; macro = unweighted mean of the eight language scores.")
# cross-check the rapidfuzz edit-distance implementation against jiwer on the test set
d = SCORED["test"]; d = d[(d.w_len > 0) & (d.hyp_n != "")]
try:
    jw, jc = jiwer.wer(d.ref_n.tolist(), d.hyp_n.tolist()), jiwer.cer(d.ref_n.tolist(), d.hyp_n.tolist())
    mine = corpus_rates(d); print(f"jiwer WER {jw:.6f} vs own {mine['wer']:.6f} | CER {jc:.6f} vs {mine['cer']:.6f}")
    assert abs(jw - mine["wer"]) < 1e-6 and abs(jc - mine["cer"]) < 1e-6
except ValueError as e: print("jiwer cross-check skipped:", e)

In [ ]:
# CELL 40 - Evaluation text normalisation (same function for reference and hypothesis; defined in Cell 35)
examples = ["नमस्ते, दुनिया! (Hello) — 2024।", "வணக்கம்; உலகம்.", "Hello, WORLD-wide!", "ఇది ఒక  పరీక్ష?"]
for e in examples: print(repr(e), "->", repr(normalize_eval(e)))
n_chk = 0
for split, d in PRED.items():
    for r in d.ref.tolist()[:2000]:
        letters = lambda s: sum(unicodedata.category(c)[0] in "LM" and not is_latin_letter(c) for c in s)
        assert letters(unicodedata.normalize("NFC", r)) == letters(normalize_eval(r)); n_chk += 1
print(f"native-script letters/marks preserved by the normaliser on {n_chk} references (no transliteration, no deletion of Indic characters)")
print("NOTE: this normaliser keeps Indic vowel signs / viramas; WERs are not comparable with numbers computed with Whisper's BasicTextNormalizer (which strips them).")
print("Empty references after normalisation:", {k: int((v.w_len == 0).sum()) for k, v in SCORED.items()})

In [ ]:
# CELL 41 - Bootstrap 95 % confidence intervals (B = CFG BOOTSTRAP_B), sentence-cluster resampling
# FLEURS has up to 3 speaker recordings of the SAME sentence, which are strongly correlated. The bootstrap therefore resamples whole (language, sentence) CLUSTERS,
# not single utterances (speaker ids are not exposed, so there is no speaker-level bootstrap). Descriptive intervals, not a hypothesis test.
rng = np.random.default_rng(CFG["SEED"])
CLUSTER_OF = dict(zip(meta_final.id, meta_final.lang + ":" + meta_final.sent_id.astype(str)))

def _arr(df, metric):
    d = df[df.w_len > 0].set_index("id")
    return (d.w_err, d.w_len) if metric == "WER" else (d.c_err, d.c_len)

def cluster_sums(ids, *vals):
    codes, uniq = pd.factorize(pd.Series([CLUSTER_OF[i] for i in ids]))
    return [np.bincount(codes, weights=np.asarray(v, float), minlength=len(uniq)) for v in vals]

def boot_draws(err_c, ln_c, B):
    n = len(err_c); idx = rng.integers(0, n, size=(B, n)); return err_c[idx].sum(1) / ln_c[idx].sum(1)

ci_rows = []
for split in EVAL_SETS:
    for metric in ("WER", "CER"):
        per_lang_draws = {}
        for scope in LANGS + ["ALL"]:
            d = SCORED[split]; d = d if scope == "ALL" else d[d.lang == scope]
            e, ln = _arr(d, metric); e_c, l_c = cluster_sums(e.index, e.values, ln.values)
            draws = boot_draws(e_c, l_c, CFG["BOOTSTRAP_B"]); point = e_c.sum() / l_c.sum()
            if scope != "ALL": per_lang_draws[scope] = draws
            ci_rows.append(dict(language=LANG_NAME.get(scope, POOLED), dataset="fleurs", split=split, metric=metric, point=point, lower_95=np.percentile(draws, 2.5), upper_95=np.percentile(draws, 97.5), n=len(e), n_clusters=len(e_c)))
        macro_draws = np.mean([per_lang_draws[l] for l in LANGS], axis=0)                                         # macro-average: resample inside every language, then average
        ci_rows.append(dict(language=MACRO, dataset="fleurs", split=split, metric=metric, point=float(np.mean([r["point"] for r in ci_rows if r["split"] == split and r["metric"] == metric and r["language"] in LANG_NAME.values()])),
                            lower_95=np.percentile(macro_draws, 2.5), upper_95=np.percentile(macro_draws, 97.5), n=len(SCORED[split]), n_clusters=int(sum(r["n_clusters"] for r in ci_rows if r["split"] == split and r["metric"] == metric and r["language"] in LANG_NAME.values()))))
CI_TABLE = pd.DataFrame(ci_rows)
print("95 % sentence-cluster bootstrap intervals (test):"); display(CI_TABLE.query("split == 'test'").round(4))

In [ ]:
prev = CFG["PREVIOUS_DENSE_198M"]
cw = CI_TABLE.query("split == 'test' and metric == 'WER' and language == @POOLED").iloc[0]; cc = CI_TABLE.query("split == 'test' and metric == 'CER' and language == @POOLED").iloc[0]
REFERENCE_COMPARISON = dict(
    reference=prev,
    wer=dict(new=100 * cw.point, ci95=[100 * cw.lower_95, 100 * cw.upper_95], previous=prev["test_wer"], abs_diff_pts=100 * cw.point - prev["test_wer"], rel_change_pct=100 * (100 * cw.point - prev["test_wer"]) / prev["test_wer"],
             previous_inside_new_ci=bool(100 * cw.lower_95 <= prev["test_wer"] <= 100 * cw.upper_95)),
    cer=dict(new=100 * cc.point, ci95=[100 * cc.lower_95, 100 * cc.upper_95], previous=prev["test_cer"], abs_diff_pts=100 * cc.point - prev["test_cer"], rel_change_pct=100 * (100 * cc.point - prev["test_cer"]) / prev["test_cer"],
             previous_inside_new_ci=bool(100 * cc.lower_95 <= prev["test_cer"] <= 100 * cc.upper_95)),
    trainable_params_ratio=PARAM_COUNTS["model"]["trainable"] / prev["trainable_params"], train_time_ratio=RUNS[TAG]["train_seconds"] / 60 / prev["train_minutes"])
for k in ("wer", "cer"):
    r = REFERENCE_COMPARISON[k]
    print(f"pooled test {k.upper()}: baseline Dense-198M {r['previous']:.2f} % -> {MODEL_NAME} {r['new']:.2f} % [95 % CI {r['ci95'][0]:.2f}, {r['ci95'][1]:.2f}]  ({r['abs_diff_pts']:+.2f} pts, {r['rel_change_pct']:+.1f} % relative); baseline value inside the new interval: {r['previous_inside_new_ci']}")
print(f"trainable parameters x{REFERENCE_COMPARISON['trainable_params_ratio']:.2f} | training time x{REFERENCE_COMPARISON['train_time_ratio']:.2f} of the baseline run")
print("The baseline is a single earlier run on the same FLEURS test set; its per-utterance outputs are not available, so this is a descriptive comparison, not a paired test. "
      "The change bundles several modifications (more trainable layers, wider decoder FFNs, SpecAugment, 2 epochs, checkpoint selection) and cannot attribute the gain to any single one.")

In [ ]:
SCRIPT_RANGES = {name: (a, b) for name, a, b in CFG["LANG_SCRIPT"].values()}
def script_of(ch):
    o = ord(ch)
    for name, (a, b) in SCRIPT_RANGES.items():
        if a <= o <= b: return name
    return "Latin" if is_latin_letter(ch) else "Other"

def script_profile(text):
    return Counter(script_of(ch) for ch in text if unicodedata.category(ch)[0] in "LM")      # letters + combining marks only

def has_repetition_loop(t):
    w = t.split()
    for n in range(1, 9):                                    # a word n-gram repeated >= 4 times in a row
        for i in range(0, len(w) - 4 * n + 1):
            if all(w[i + j * n: i + (j + 1) * n] == w[i: i + n] for j in range(1, 4)): return True
    return re.search(r"(.{6,}?)\1{3,}", t.replace(" ", "")) is not None                      # un-spaced character loops

def failure_flags(df):
    d = df.copy()
    prof = d.hyp_n.map(script_profile)
    tot = prof.map(lambda c: sum(c.values()))
    d["script_frac"] = [(c[CFG["LANG_SCRIPT"][l][0]] / t) if t else np.nan for c, t, l in zip(prof, tot, d.lang)]
    d["dominant_script"] = [c.most_common(1)[0][0] if t else None for c, t in zip(prof, tot)]
    d["f_empty"] = d.hyp_n == ""
    d["f_repetition"] = d.hyp_n.map(has_repetition_loop)
    d["f_wrong_script"] = (tot > 0) & (d.script_frac < CFG["SCRIPT_OK_THRESHOLD"])
    d["f_unfinished"] = ~d.finished_by_eos
    d["f_excessive"] = d.hyp_n.str.len() > 2 * d.ref_n.str.len()          # crude heuristic: normalised hypothesis has > 2x the characters of the reference
    d["f_any"] = d[["f_empty", "f_repetition", "f_wrong_script", "f_unfinished", "f_excessive"]].any(axis=1)
    return d

FLAGGED = {s: failure_flags(d) for s, d in SCORED.items()}
FT = ["f_empty", "f_repetition", "f_wrong_script", "f_unfinished", "f_excessive", "f_any"]
fail_rows = []
for split, d in FLAGGED.items():
    for l in LANGS + ["ALL"]:
        x = d if l == "ALL" else d[d.lang == l]
        fail_rows.append(dict(split=split, language=LANG_NAME.get(l, POOLED), n=len(x), **{c[2:]: int(x[c].sum()) for c in FT}, failure_rate=float(x.f_any.mean())))
FAIL_TABLE = pd.DataFrame(fail_rows)
print("Failure counts (test). Categories overlap (one utterance can carry several flags); `excessive` is a crude character-length heuristic:")
display(FAIL_TABLE.query("split == 'test'").drop(columns="split"))

In [ ]:
# CELL 43 - Script accuracy: is the transcript predominantly written in the expected script?
sc_rows = []
for split, d in FLAGGED.items():
    for l in LANGS + ["ALL"]:
        x = d if l == "ALL" else d[d.lang == l]
        nonempty = x[x.hyp_n != ""]
        correct = (nonempty.script_frac >= CFG["SCRIPT_OK_THRESHOLD"]).sum(); wrong = (nonempty.script_frac < CFG["SCRIPT_OK_THRESHOLD"]).sum()
        wrong_dom = nonempty[nonempty.script_frac < CFG["SCRIPT_OK_THRESHOLD"]].dominant_script.value_counts().to_dict()
        sc_rows.append(dict(split=split, language=LANG_NAME.get(l, POOLED), expected_script=CFG["LANG_SCRIPT"][l][0] if l in LANGS else "-",
                            n=len(x), correct_script_pct=100 * correct / max(1, len(x)), wrong_script_pct=100 * wrong / max(1, len(x)),
                            empty_pct=100 * (len(x) - len(nonempty)) / max(1, len(x)), mean_expected_script_fraction=nonempty.script_frac.mean(), wrong_script_dominant=wrong_dom))
SCRIPT_TABLE = pd.DataFrame(sc_rows)
display(SCRIPT_TABLE.query("split == 'test'").drop(columns="split").round(2))
print("Script accuracy = share of utterances whose letters are >= 50 % in the expected script (a predominant-script rate, an operational definition).")
print("Caveat: Hindi and Marathi share the Devanagari script, so script accuracy cannot detect Hindi<->Marathi confusions; WER captures those.")

In [ ]:
_rs = np.random.RandomState(CFG["SEED"])
d = FLAGGED["test"]
for cat in ("f_empty", "f_repetition", "f_wrong_script", "f_unfinished", "f_excessive"):
    x = d[d[cat]]
    print(f"\n--- {MODEL_NAME} / {cat[2:]}: {len(x)} test utterances ---")
    if len(x): display(x.iloc[_rs.choice(len(x), size=min(3, len(x)), replace=False)][["id", "lang", "ref", "hyp"]].assign(ref=lambda t: t.ref.str[:100], hyp=lambda t: t.hyp.str[:100]))

tok_rows = []
for l in LANGS:
    t = tables["test"][tables["test"].lang == l]; chars = t.text.str.len(); ntok_ = t.text_ids.map(len)
    tr_ = tok_report[(tok_report.split == "test") & (tok_report.lang == l)].iloc[0]
    tok_rows.append(dict(language=LANG_NAME[l], n=len(t), mean_chars=chars.mean(), mean_tokens=ntok_.mean(), tokens_per_char=ntok_.sum() / chars.sum(), chars_per_token=chars.sum() / ntok_.sum(),
                         tokens_p10=ntok_.quantile(.1), tokens_p50=ntok_.median(), tokens_p90=ntok_.quantile(.9), removed_by_440_cap_pct=float(tr_["removal_%"])))
TOKEN_DF = pd.DataFrame(tok_rows).set_index("language")
TOKEN_DF["WER%"] = [RATE_TABLES["test"].loc[LANG_NAME[l], "WER %"] for l in LANGS]; TOKEN_DF["CER%"] = [RATE_TABLES["test"].loc[LANG_NAME[l], "CER %"] for l in LANGS]
display(TOKEN_DF.round(3))
SPEARMAN = {}
for col in ("WER%", "CER%"):
    rho, p = spearmanr(TOKEN_DF["tokens_per_char"], TOKEN_DF[col]); SPEARMAN[f"tokens_per_char vs {col}"] = dict(rho=float(rho), p=float(p), n_languages=len(TOKEN_DF))
print("Spearman correlation of tokenisation difficulty (Whisper tokens per character, FLEURS test transcripts) with error, across the 8 languages:")
display(pd.DataFrame(SPEARMAN).T.round(3))
print("Caveat: n = 8 languages, so the correlation has very low power and confounds script, morphology and data amount; it is descriptive, not causal. No tokenizer / model was trained.")

In [ ]:
# CELL 43c - WER by language and by utterance length (validation = development, test = final): pooled duration / token-length quintiles and language-specific terciles
def grouped_rates(d, by):
    g = d.groupby(by, observed=True).agg(n=("id", "size"), mean_audio_s=("audio_sec", "mean"), mean_ref_tokens=("ref_tokens", "mean"), w_err=("w_err", "sum"), w_len=("w_len", "sum"),
                                         c_err=("c_err", "sum"), c_len=("c_len", "sum"), eos=("finished_by_eos", "mean"))
    g["WER %"] = 100 * g.w_err / g.w_len.clip(lower=1); g["CER %"] = 100 * g.c_err / g.c_len.clip(lower=1); g["finished_by_eos %"] = 100 * g.eos
    return g[["n", "mean_audio_s", "mean_ref_tokens", "WER %", "CER %", "finished_by_eos %"]]

LENGTH_TABLES = {}
for split in EVAL_SETS:
    d = SCORED[split].copy()
    d["ref_tokens"] = d.id.map(dict(zip(tables[split].id, tables[split].text_ids.map(len)))).astype(float)
    d["audio_quintile"] = pd.qcut(d.audio_sec, 5, duplicates="drop"); d["token_quintile"] = pd.qcut(d.ref_tokens, 5, duplicates="drop")
    d["lang_audio_tercile"] = d.groupby("lang").audio_sec.transform(lambda s: pd.qcut(s, 3, labels=["short", "medium", "long"]))
    by_audio, by_tok = grouped_rates(d, "audio_quintile"), grouped_rates(d, "token_quintile")
    by_lang_terc = grouped_rates(d, ["lang", "lang_audio_tercile"])["WER %"].unstack("lang_audio_tercile").rename(index=LANG_NAME)
    LENGTH_TABLES[split] = dict(by_audio_duration=by_audio, by_reference_tokens=by_tok, by_language_x_duration_tercile=by_lang_terc)
    print(f"\n=== {split}: WER / CER by audio-duration quintile (pooled over languages) ==="); display(by_audio.round(2))
    print(f"=== {split}: by number of reference Whisper tokens (quintiles) ==="); display(by_tok.round(2))
    print(f"=== {split}: WER % by language and language-specific duration tercile ==="); display(by_lang_terc.round(2))
print("The 440-token cap removes the longest transcripts of the token-hungry languages (Cell 13), so the longest bins are under-represented for those languages.")

In [ ]:
# CELL 44 - Inference speed (steady state; model loading reported separately; features pre-extracted, host->device copy included)
speed_rows_idx = []
for l in LANGS:
    ix = np.where(np.array(test_pool.lang) == l)[0]
    speed_rows_idx += sorted(np.random.RandomState(CFG["SEED"]).choice(ix, size=CFG["SPEED_UTTS_PER_LANG"], replace=False).tolist())
speed_pool = FleursPool(tables["test"].iloc[speed_rows_idx].reset_index(drop=True), test_ds)
speed_batches = [eval_collate([speed_pool[i] for i in b]) for b in make_lang_batches(speed_pool, CFG["SPEED_UTTS_PER_LANG"])]

@torch.no_grad()
def time_generate(model, bs):
    n_audio = n_tok = 0
    torch.cuda.synchronize(); t0 = time.time()
    for b in speed_batches:
        for j in range(0, b["input_features"].shape[0], bs):
            f = b["input_features"][j:j + bs].to(DEVICE)
            with torch.autocast("cuda", dtype=torch.float16):
                seq = model.generate(input_features=f, language=b["langs"][0], task="transcribe", return_timestamps=False,
                                     max_new_tokens=CFG["MAX_NEW_TOKENS"], do_sample=False, num_beams=1).cpu()
            n_audio += sum(b["audio_sec"][j:j + bs])
            for row in seq:
                c, fin = cut_at_eos(row); n_tok += int((c < EOS_ID).sum()) + int(fin)     # ids < EOS are exactly the text tokens (Cell 16 asserts this); +1 for the EOS itself
    torch.cuda.synchronize(); dt = time.time() - t0
    return dict(total_audio_sec=n_audio, wall_sec=dt, rtf=dt / n_audio, tokens=n_tok, tokens_per_sec=n_tok / dt)

speed_rows = []
model = load_best_model()
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):      # warm-up (excluded from timing)
    model.generate(input_features=speed_batches[0]["input_features"][:2].to(DEVICE), language=speed_batches[0]["langs"][0], task="transcribe", max_new_tokens=16)
for bs in (1, 16):
    r = time_generate(model, bs); speed_rows.append(dict(model=MODEL_NAME, batch_size=bs, model_load_sec=LOAD_SECONDS[SELECTED_STEP], **r))
    print(f"[{MODEL_NAME}] bs={bs:2d}: {r['total_audio_sec']:.0f}s audio in {r['wall_sec']:.1f}s -> RTF {r['rtf']:.4f} | {r['tokens_per_sec']:.1f} tok/s")
del model; gc.collect(); torch.cuda.empty_cache()
SPEED_TABLE = pd.DataFrame(speed_rows); display(SPEED_TABLE.round(4))
print("RTF = ASR model inference only: log-mel features are pre-extracted (not timed); host->device copy + generate() are timed. Not an end-to-end latency figure.")

In [ ]:
Te, Td, Vv = 1500, 217, base_cpu.config.vocab_size                     # encoder frames; mean target tokens (FLEURS Indic average, Whisper tokenizer)
def _enc_fwd(w): return 8 * Te * D_MODEL * D_MODEL + 4 * Te * Te * D_MODEL + 4 * Te * D_MODEL * w
def _dec_fwd(w): return (8 * Td * D_MODEL ** 2 + 4 * Td * Td * D_MODEL) + (4 * Td * D_MODEL ** 2 + 4 * Te * D_MODEL ** 2 + 4 * Td * Te * D_MODEL) + 4 * Td * D_MODEL * w
def train_gflops_per_sample(enc_from0, dec_from0, w_enc, w_dec):
    """fwd + bwd FLOPs per training sample (multiply-add = 2 FLOPs). Trainable layers: dgrad + wgrad; frozen decoder layers still need dgrad (cross-attention path to the encoder); frozen lower encoder layers need none."""
    nd = 12 - dec_from0; ne = 12 - enc_from0
    enc = sum(_enc_fwd(w_enc if i >= enc_from0 else H0) for i in range(12)) + sum(2 * _enc_fwd(w_enc) for _ in range(ne))
    dec = sum(_dec_fwd(w_dec if i >= dec_from0 else H0) for i in range(12)) + sum(2 * _dec_fwd(w_dec) for _ in range(nd)) + sum(_dec_fwd(H0) for _ in range(12 - nd))
    return (enc + dec + 2 * (2 * Td * D_MODEL * Vv)) / 1e9
cur_gf = train_gflops_per_sample(min(CFG["TRAIN_ENC_LAYERS"]), min(CFG["TRAIN_DEC_LAYERS"]), ENC_FFN_WIDTH, DEC_FFN_WIDTH); prev_gf = train_gflops_per_sample(8, 8, H0, 24584)
sp1 = SPEED_TABLE[SPEED_TABLE.batch_size == 1].iloc[0]; sp16 = SPEED_TABLE[SPEED_TABLE.batch_size == 16].iloc[0]
run_ = RUNS[TAG]; pcm = PARAM_COUNTS["model"]
COMPUTE_TABLE = pd.DataFrame([
    ("Total parameters", pcm["total"], CFG["PREVIOUS_DENSE_198M"]["total_params"]), ("Trainable parameters", pcm["trainable"], CFG["PREVIOUS_DENSE_198M"]["trainable_params"]), ("Frozen parameters", pcm["frozen"], CFG["PREVIOUS_DENSE_198M"]["total_params"] - CFG["PREVIOUS_DENSE_198M"]["trainable_params"]),
    ("Active parameters / token (dense: all)", pcm["total"], CFG["PREVIOUS_DENSE_198M"]["total_params"]), ("Decoder FFN parameters per widened layer", ffn_param_count(DEC_FFN_WIDTH), ffn_param_count(24584)), ("Encoder FFN parameters per widened layer", ffn_param_count(ENC_FFN_WIDTH), ffn_param_count(H0)),
    ("Training FLOPs / sample (analytic, GFLOPs)", cur_gf, prev_gf), ("Training time (min, excl. monitoring)", run_["train_seconds"] / 60, CFG["PREVIOUS_DENSE_198M"]["train_minutes"]),
    ("Training instances processed", run_["steps"] * EFFECTIVE_BS, 24000), ("Peak training GPU memory (GiB)", run_["peak_gpu_mem_gb"], np.nan), ("Analytic worst-case memory estimate (GiB)", sum(estimate_memory_gb(MICRO_BS)), np.nan),
    ("Monitoring decode during training (min)", run_["eval_seconds"] / 60, np.nan), ("Full validation decode (min, selected checkpoint)", DECODE_SECONDS.get(("validation", SELECTED_STEP), np.nan) / 60, np.nan),
    ("Full test decode (min, selected checkpoint)", DECODE_SECONDS.get(("test", SELECTED_STEP), np.nan) / 60, np.nan),
    ("Inference RTF, batch 1", sp1.rtf, np.nan), ("Inference RTF, batch 16", sp16.rtf, np.nan), ("Inference tokens/s, batch 16", sp16.tokens_per_sec, np.nan)],
    columns=["Metric", MODEL_NAME, "Baseline Dense-198M"]).set_index("Metric")
display(COMPUTE_TABLE.round(3))
print(f"analytic training cost per sample x{cur_gf / prev_gf:.2f} of the baseline run; measured time per instance x{(run_['train_seconds'] / (run_['steps'] * EFFECTIVE_BS)) / (CFG['PREVIOUS_DENSE_198M']['train_minutes'] * 60 / 24000):.2f}. "
      "A dense model activates every parameter for every token, so active parameters per token = total parameters.")

## Visualisation
One model, one colour; sequential one-hue ramps for magnitudes. All values are also available as tables above and in the exported JSON.

In [ ]:
PLOT_DIR = f"{OUT_DIR}/plots"; os.makedirs(PLOT_DIR, exist_ok=True)
INK, INK2, GRIDC = "#0b0b0b", "#52514e", "#e6e5e1"
MODEL_COL = "#2a78d6"; ACCENT = "#eb6834"
from matplotlib.colors import LinearSegmentedColormap
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"])
plt.rcParams.update({"font.size": 10, "axes.edgecolor": GRIDC, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
                     "text.color": INK, "axes.titlesize": 11, "axes.titleweight": "bold", "figure.facecolor": "white", "axes.facecolor": "white"})

def style_ax(ax, grid_axis="y"):
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    ax.grid(axis=grid_axis, color=GRIDC, linewidth=0.8); ax.set_axisbelow(True)

def save_fig(fig, name):
    fig.savefig(f"{PLOT_DIR}/{name}.png", dpi=150, bbox_inches="tight"); plt.show()

def heat(ax, M, rows, cols, fmt="{:.2f}", vmin=None, vmax=None, title=""):
    M = np.asarray(M, float); vmin = np.nanmin(M) if vmin is None else vmin; vmax = np.nanmax(M) if vmax is None else vmax
    im = ax.imshow(M, cmap=SEQ, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=35, ha="right"); ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, fmt.format(M[i, j]), ha="center", va="center", fontsize=8, color=("white" if (M[i, j] - vmin) / max(1e-12, vmax - vmin) > 0.55 else INK))
    for s in ax.spines.values(): s.set_visible(False)
    ax.set_title(title, loc="left"); ax.tick_params(length=0)
    return im

lg, vl = RUNS[TAG]["log"], RUNS[TAG]["val"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(lg.step, lg.loss, color=MODEL_COL, lw=2); axes[1].plot(vl.step, vl.val_loss, color=MODEL_COL, lw=2, marker="o", ms=5); axes[2].plot(vl.step, 100 * vl.val_wer, color=MODEL_COL, lw=2, marker="o", ms=5)
for ax, t, yl in zip(axes, ["Training loss (token-mean cross-entropy)", "Validation loss (fixed subset)", "Validation WER % (fixed subset)"], ["cross-entropy", "cross-entropy", "WER %"]):
    style_ax(ax); ax.set_title(t, loc="left"); ax.set_xlabel("optimizer step")
    ax.set_ylabel(yl)
    for e in range(1, CFG["NUM_EPOCHS"]): ax.axvline(e * STEPS_PER_EPOCH, color=GRIDC, lw=1.5, ls="--")
    ax.axvline(SELECTED_STEP, color=ACCENT, lw=1.2, ls=":")
axes[2].annotate(f"selected: step {SELECTED_STEP}", (SELECTED_STEP, 100 * vl.val_wer.min()), xytext=(5, 12), textcoords="offset points", fontsize=8, color=ACCENT)
fig.tight_layout(); save_fig(fig, "51_training_curves")
fig, ax = plt.subplots(figsize=(8.5, 4)); cmap_ = plt.get_cmap("tab10")
for k, l in enumerate(LANGS): ax.plot(vl.step, 100 * vl[f"wer_{l}"], marker="o", ms=3, lw=1.5, color=cmap_(k), label=LANG_NAME[l])
style_ax(ax); ax.set_xlabel("optimizer step"); ax.set_ylabel("validation-subset WER %"); ax.set_title("Per-language validation-subset WER during training (25 utterances / language)", loc="left", fontsize=10); ax.legend(frameon=False, ncol=4, fontsize=8)
fig.tight_layout(); save_fig(fig, "51b_per_language_validation_curves")

In [ ]:
names_full = [LANG_NAME[l] for l in LANGS] + [POOLED, MACRO]; names_short = [LANG_NAME[l] for l in LANGS] + ["Pooled", "Macro"]
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
for ax, metric, prev_val in zip(axes, ("WER", "CER"), (CFG["PREVIOUS_DENSE_198M"]["test_wer"], CFG["PREVIOUS_DENSE_198M"]["test_cer"])):
    d = CI_TABLE.query("split == 'test' and metric == @metric").set_index("language").loc[names_full]
    x = np.arange(len(names_full))
    ax.bar(x, 100 * d.point, 0.65, color=[MODEL_COL] * len(LANGS) + [ACCENT, ACCENT])
    ax.errorbar(x, 100 * d.point, yerr=[100 * (d.point - d.lower_95), 100 * (d.upper_95 - d.point)], fmt="none", ecolor=INK2, elinewidth=1, capsize=2)
    ax.axhline(prev_val, color=INK2, lw=1, ls="--"); ax.annotate(f"baseline Dense-198M pooled {prev_val:.2f} %", (len(names_full) - 0.5, prev_val), ha="right", va="bottom", fontsize=8, color=INK2)
    ax.set_xticks(x); ax.set_xticklabels(names_short, rotation=30, ha="right"); style_ax(ax); ax.set_ylabel(f"{metric} %"); ax.set_title(f"{MODEL_NAME}: {metric} by language - FLEURS test (95% bootstrap CI)", loc="left")
fig.tight_layout(); save_fig(fig, "52_language_performance")

In [ ]:
rows_ = [LANG_NAME[l] for l in LANGS] + ["Pooled", "Macro"]
fig, axes = plt.subplots(1, 2, figsize=(9, 4.6))
for ax, metric in zip(axes, ("WER %", "CER %")):
    M = np.array([[RATE_TABLES[s].loc[n, metric] for s in ("validation", "test")] for n in [LANG_NAME[l] for l in LANGS] + [POOLED, MACRO]])
    heat(ax, M, rows_, ["Validation", "Test"], fmt="{:.1f}", title=f"Language x {metric[:-2]} (%)")
fig.tight_layout(); save_fig(fig, "53_dataset_performance")

In [ ]:
ftypes = ["empty", "repetition", "wrong_script", "unfinished", "excessive"]
d = FAIL_TABLE.query("split == 'test'").set_index("language")
M = np.array([[100 * d.loc[LANG_NAME[l], f] / d.loc[LANG_NAME[l], "n"] for f in ftypes] for l in LANGS])
fig, ax = plt.subplots(figsize=(6.5, 4.6))
heat(ax, M, [LANG_NAME[l] for l in LANGS], ftypes, fmt="{:.1f}", vmin=0, vmax=max(1.0, M.max()), title=f"{MODEL_NAME} - failure rate (% of test utterances; categories overlap)")
fig.tight_layout(); save_fig(fig, "56_failure_analysis")

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for split, col in (("validation", ACCENT), ("test", MODEL_COL)):
    t = LENGTH_TABLES[split]["by_audio_duration"]; axes[0].plot(range(len(t)), t["WER %"], marker="o", color=col, lw=2, label=split)
axes[0].set_xticks(range(len(t))); axes[0].set_xticklabels([f"{m_:.1f}s" for m_ in LENGTH_TABLES["test"]["by_audio_duration"].mean_audio_s], fontsize=8); axes[0].legend(frameon=False)
axes[0].set_xlabel("audio-duration quintile (mean duration)"); axes[0].set_ylabel("WER %"); axes[0].set_title("WER by utterance length (pooled)", loc="left"); style_ax(axes[0])
heat(axes[1], LENGTH_TABLES["test"]["by_language_x_duration_tercile"].values, list(LENGTH_TABLES["test"]["by_language_x_duration_tercile"].index), list(LENGTH_TABLES["test"]["by_language_x_duration_tercile"].columns), fmt="{:.1f}", title="Test WER % by language x duration tercile")
axes[2].scatter(TOKEN_DF.tokens_per_char, TOKEN_DF["WER%"], color=MODEL_COL, s=40, zorder=3)
for lang_, r_ in TOKEN_DF.iterrows(): axes[2].annotate(lang_, (r_.tokens_per_char, r_["WER%"]), fontsize=8, xytext=(4, 3), textcoords="offset points", color=INK2)
sp_ = SPEARMAN["tokens_per_char vs WER%"]; style_ax(axes[2], "both"); axes[2].set_xlabel("Whisper tokens per character (test transcripts)"); axes[2].set_ylabel("test WER %")
axes[2].set_title(f"Tokenisation vs WER: Spearman rho = {sp_['rho']:.2f} (p = {sp_['p']:.2f}, n = 8)", loc="left", fontsize=9)
fig.tight_layout(); save_fig(fig, "56b_length_and_tokenisation")

## Exported artifacts
`experiment_config.json`, `results.json`, `transcripts.jsonl`, `whisper_dense_checkpoint/` (selected checkpoint: trained parameter values + config), `dense_final/` (last step) and `plots/` are written to the output directory.

In [ ]:
def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, pd.DataFrame): return jsonable(o.reset_index().to_dict(orient="records") if (o.index.name or isinstance(o.index, pd.MultiIndex)) else o.to_dict(orient="records"))
    if isinstance(o, np.ndarray): return jsonable(o.tolist())
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, (np.floating, float)): v = float(o); return v if np.isfinite(v) else None
    if isinstance(o, (np.integer,)): return int(o)
    return o

def pkg_versions():
    out = {}
    for m_ in ("torch", "transformers", "datasets", "huggingface_hub", "accelerate", "jiwer", "safetensors", "tokenizers", "soundfile", "librosa"):
        try: out[m_] = md.version(m_)
        except Exception: out[m_] = "?"
    return out

results = dict(
    experiment="Function-Preserving Scaling of Whisper for Multilingual Indic ASR (single dense model)",
    model_name=MODEL_NAME, model=CFG["MODEL_ID"], model_revision=MODEL_REVISION, dataset=CFG["DATASET_ID"], dataset_configs=CFG["LANG_CONFIGS"], languages={l: LANG_NAME[l] for l in LANGS},
    experiment_configuration=EXPERIMENT_CONFIG, environment=dict(packages=pkg_versions(), cuda=torch.version.cuda, gpu=GPU_NAME, vram_gib=GPU_VRAM_GB), licenses=CFG["LICENSES"],
    training_configuration={k: v for k, v in CFG.items() if k not in ("LANG_SCRIPT",)},
    parameter_counts=PARAM_COUNTS, initialisation_equivalence_tests=EQUIV_RESULT, compute_and_parameters=COMPUTE_TABLE,
    dataset_statistics=stat_table, data_funnel=FUNNEL, filtering=dict(duration=dur_report, token_length=tok_report),
    training=dict(train_seconds=RUNS[TAG]["train_seconds"], monitoring_seconds=RUNS[TAG]["eval_seconds"], avg_step_seconds=RUNS[TAG]["avg_step_seconds"], peak_gpu_mem_gb=RUNS[TAG]["peak_gpu_mem_gb"],
                  steps_run=RUNS[TAG]["steps"], total_steps=TOTAL_STEPS, epochs_completed=RUNS[TAG]["epochs_completed"], examples_processed=RUNS[TAG]["examples_processed"], throughput_examples_per_sec=RUNS[TAG]["throughput_examples_per_sec"], early_stopped=RUNS[TAG].get("early_stopped"), oom_events=RUNS[TAG]["oom_events"], skipped_fp16_steps=RUNS[TAG]["skipped_optimizer_steps"],
                  validation_curve=RUNS[TAG]["val"]),
    checkpoint_selection=SELECTION, validation_results=RATE_TABLES["validation"], test_results=RATE_TABLES["test"],
    confidence_intervals=CI_TABLE, reference_comparison=REFERENCE_COMPARISON, script_accuracy=SCRIPT_TABLE, failure_rates=FAIL_TABLE,
    wer_by_length={s: {k: v for k, v in t.items()} for s, t in LENGTH_TABLES.items()}, tokenization_analysis=dict(per_language=TOKEN_DF, spearman=SPEARMAN), inference_speed=SPEED_TABLE,
    experiment_hash=EXPERIMENT_HASH, experiment_fingerprint=FINGERPRINT, notebook_runtime_seconds=time.time() - NOTEBOOK_T0)
def write_results():
    with open(f"{OUT_DIR}/results.json", "w", encoding="utf-8") as f: json.dump(jsonable(results), f, indent=1, ensure_ascii=False)
write_results(); print("wrote", f"{OUT_DIR}/results.json", f"({os.path.getsize(OUT_DIR + '/results.json')/1e3:.0f} kB)")
EXPERIMENT_CONFIG.update(selected_checkpoint_step=SELECTED_STEP, training_seconds=RUNS[TAG]["train_seconds"], notebook_runtime_seconds=time.time() - NOTEBOOK_T0,
                         test_wer_pooled_pct=REFERENCE_COMPARISON["wer"]["new"], test_cer_pooled_pct=REFERENCE_COMPARISON["cer"]["new"])
with open(f"{OUT_DIR}/experiment_config.json", "w") as f: json.dump(jsonable(EXPERIMENT_CONFIG), f, indent=1)

In [ ]:
with open(f"{OUT_DIR}/transcripts.jsonl", "w", encoding="utf-8") as f:
    n = 0
    for split, d in FLAGGED.items():
        for r in d.itertuples():
            f.write(json.dumps(dict(id=r.id, dataset="fleurs", lang=r.lang, audio_sec=float(r.audio_sec), ref=r.ref, hyp=r.hyp,
                                    wer=None if not np.isfinite(r.wer) else float(r.wer), cer=None if not np.isfinite(r.cer) else float(r.cer),
                                    finished_by_eos=bool(r.finished_by_eos), model=MODEL_NAME, split=split), ensure_ascii=False) + "\n"); n += 1
print("wrote", n, "lines to transcripts.jsonl")

In [ ]:
from safetensors import safe_open
EXPORT_DIR = f"{OUT_DIR}/whisper_dense_checkpoint"; os.makedirs(EXPORT_DIR, exist_ok=True)
src_ = ckpt_path(TAG, SELECTED_STEP)
shutil.copy(f"{src_}/trainable.safetensors", f"{EXPORT_DIR}/model.safetensors")
with open(f"{src_}/config.json") as f: cfg_ = json.load(f)
with safe_open(f"{EXPORT_DIR}/model.safetensors", framework="pt") as f_: saved_names = sorted(f_.keys())
cfg_.update(format="trainable-parameter values (safetensors); NOT a difference to the pretrained weights", trainable_names=saved_names, base_model=CFG["MODEL_ID"], model_revision=MODEL_REVISION,
            original_ffn_dim=H0, architecture=dict(original_ffn_dim=H0, encoder_widened_ffn_dim=ENC_FFN_WIDTH, decoder_widened_ffn_dim=DEC_FFN_WIDTH, widened_encoder_layers=[i + 1 for i in CFG["TRAIN_ENC_LAYERS"]],
                                                   widened_decoder_layers=[i + 1 for i in CFG["TRAIN_DEC_LAYERS"]], trainable_parameters=PARAM_COUNTS["model"]["trainable"], total_parameters=PARAM_COUNTS["model"]["total"]), environment=pkg_versions(), cuda=torch.version.cuda, tokenizer_revision=MODEL_REVISION, licenses=CFG["LICENSES"],
            experiment_hash=EXPERIMENT_HASH, selected_step=SELECTED_STEP, checkpoint_selection=SELECTION)
with open(f"{EXPORT_DIR}/config.json", "w") as f: json.dump(jsonable(cfg_), f, indent=1)
print(f"{MODEL_NAME}: checkpoint {os.path.getsize(EXPORT_DIR + '/model.safetensors')/1e6:.0f} MB, {len(saved_names)} tensors -> {EXPORT_DIR}   (last-step checkpoint: {OUT_DIR}/{TAG}_final)")

In [ ]:
def load_whisper_dense(ckpt_dir=EXPORT_DIR, device=DEVICE):
    t0 = time.time()
    with open(f"{ckpt_dir}/config.json") as f: c = json.load(f)
    rev = dict(revision=c["model_revision"]) if c.get("model_revision") else {}
    model = WhisperForConditionalGeneration.from_pretrained(c["base_model"], **rev)
    model.config.forced_decoder_ids = None; model.generation_config.forced_decoder_ids = None; model.generation_config.max_length = None
    for stack, ids, width in ((model.model.encoder, c["widened_enc_layers"], c["enc_ffn_width"]), (model.model.decoder, c["widened_dec_layers"], c["dec_ffn_width"])):   # 2. rebuild the widened FFNs (values are overwritten by the checkpoint)
        for i in ids:
            L = stack.layers[i]; L.fc1, L.fc2 = nn.Linear(c["hidden_size"], width), nn.Linear(width, c["hidden_size"])
    sd = load_file(f"{ckpt_dir}/model.safetensors")
    assert set(sd) == set(c["trainable_names"]), "checkpoint tensors differ from the recorded trainable-parameter list"
    own = dict(model.named_parameters()); assert all(n in own for n in sd), "checkpoint contains tensors the rebuilt model does not have"
    res = model.load_state_dict(sd, strict=False)
    assert not res.unexpected_keys, res.unexpected_keys
    assert not [n for n in res.missing_keys if n in sd]
    proc = WhisperProcessor.from_pretrained(c["base_model"], **rev)
    model = model.to(device).eval()
    load_whisper_dense.last_load_seconds = time.time() - t0
    return model, proc
print("load_whisper_dense() defined")

In [ ]:
rec_model, rec_proc = load_whisper_dense()
print(f"[{MODEL_NAME}] reconstructed in {load_whisper_dense.last_load_seconds:.1f}s | device: {next(rec_model.parameters()).device} | eval mode: {not rec_model.training} | parameters: {count_params(rec_model)['total']:,}")
u = test_pool[0]; lang0 = u["lang"]
f_ = rec_proc.feature_extractor(u["audio"], sampling_rate=16000, return_tensors="pt").input_features.to(DEVICE)
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    out_ids = rec_model.generate(input_features=f_, language=lang0, task="transcribe", return_timestamps=False, max_new_tokens=CFG["MAX_NEW_TOKENS"], do_sample=False, num_beams=1)
hyp0 = rec_proc.batch_decode(out_ids.cpu().tolist(), skip_special_tokens=True)[0].strip()
print(f"[{lang0}] REF: {u['transcription']}\n[{lang0}] HYP: {hyp0}"); assert len(hyp0) > 0
b0 = collator([test_pool[0], test_pool[1]])
ref_model = load_best_model()
with torch.no_grad():
    d_ = (rec_model(input_features=b0["input_features"].to(DEVICE), decoder_input_ids=b0["decoder_input_ids"].to(DEVICE)).logits -
          ref_model(input_features=b0["input_features"].to(DEVICE), decoder_input_ids=b0["decoder_input_ids"].to(DEVICE)).logits).abs().max().item()
print(f"[{MODEL_NAME}] max |logit difference| reconstructed vs in-notebook selected model: {d_:.2e}"); assert d_ < 1e-4
del rec_model, ref_model; gc.collect(); torch.cuda.empty_cache()
print("FINAL ARTIFACT VERIFICATION: PASS")

In [ ]:
# CELL 64 - Final tables (pooled/micro vs macro averages are labelled explicitly; script accuracy / failure rate are REPORTED metrics, no hard pass/fail thresholds)
def final_table(split):
    rows = []
    for l in LANGS + ["ALL"]:
        nm = LANG_NAME.get(l, POOLED); rt = RATE_TABLES[split].loc[nm]
        sc = SCRIPT_TABLE.query("split == @split and language == @nm").iloc[0]; fr = FAIL_TABLE.query("split == @split and language == @nm").iloc[0]
        ci_w = CI_TABLE.query("split == @split and metric == 'WER' and language == @nm").iloc[0]; ci_c = CI_TABLE.query("split == @split and metric == 'CER' and language == @nm").iloc[0]
        rows.append({"Language": nm, "WER %": rt["WER %"], "WER 95% CI": f"[{100*ci_w.lower_95:.1f}, {100*ci_w.upper_95:.1f}]", "CER %": rt["CER %"], "CER 95% CI": f"[{100*ci_c.lower_95:.1f}, {100*ci_c.upper_95:.1f}]",
                     "Script acc %": sc.correct_script_pct, "Failure rate %": 100 * fr.failure_rate, "n": int(rt["n"])})
    t = pd.DataFrame(rows).set_index("Language"); t.loc[MACRO] = [RATE_TABLES[split].loc[MACRO, "WER %"], "", RATE_TABLES[split].loc[MACRO, "CER %"], "", np.nan, np.nan, int(RATE_TABLES[split].loc[MACRO, "n"])]
    return t
for split in ("test", "validation"):
    print(f"\n=== {MODEL_NAME} - FLEURS {split} ({'FINAL held-out' if split == 'test' else 'development / checkpoint selection'}) ==="); display(final_table(split).round(2))
rt = RATE_TABLES["test"]; pooled, macro = rt.loc[POOLED], rt.loc[MACRO]
print(f"\nPooled (micro) test WER {pooled['WER %']:.2f} % / CER {pooled['CER %']:.2f} %   |   macro-average test WER {macro['WER %']:.2f} % / CER {macro['CER %']:.2f} %")
_sc = SCRIPT_TABLE.query("split == 'test' and language == @POOLED").iloc[0].correct_script_pct; _fr = 100 * FAIL_TABLE.query("split == 'test' and language == @POOLED").iloc[0].failure_rate
print(f"Baseline Dense-198M : pooled WER {CFG['PREVIOUS_DENSE_198M']['test_wer']:.2f} % / CER {CFG['PREVIOUS_DENSE_198M']['test_cer']:.2f} % / script acc {CFG['PREVIOUS_DENSE_198M']['script_acc']:.2f} % / failure rate {CFG['PREVIOUS_DENSE_198M']['failure_rate']:.2f} %")
print(f"{MODEL_NAME:19s} : pooled WER {pooled['WER %']:.2f} % / CER {pooled['CER %']:.2f} % / script acc {_sc:.2f} % / failure rate {_fr:.2f} %   (change {REFERENCE_COMPARISON['wer']['abs_diff_pts']:+.2f} / {REFERENCE_COMPARISON['cer']['abs_diff_pts']:+.2f} pts WER / CER)")
print(f"\nTrainable / total parameters : {PARAM_COUNTS['model']['trainable']:,} / {PARAM_COUNTS['model']['total']:,} ({MODEL_NAME})")
print(f"Training time                : {RUNS[TAG]['train_seconds']/60:.1f} min (+{RUNS[TAG]['eval_seconds']/60:.1f} min monitoring) | steps run {RUNS[TAG]['steps']}/{TOTAL_STEPS} ({RUNS[TAG]['epochs_completed']:.2f} epochs, {RUNS[TAG]['examples_processed']:,} examples, {RUNS[TAG]['throughput_examples_per_sec']:.1f} ex/s) | peak GPU {RUNS[TAG]['peak_gpu_mem_gb']:.2f} GB | selected checkpoint step {SELECTED_STEP}")
print(f"Inference (model only)       : RTF {sp1.rtf:.4f} (batch 1) / {sp16.rtf:.4f} (batch 16), {sp16.tokens_per_sec:.0f} tokens/s (batch 16)")
print(f"Total notebook runtime       : {(time.time()-NOTEBOOK_T0)/3600:.2f} h")

In [ ]:
rw, rc = REFERENCE_COMPARISON["wer"], REFERENCE_COMPARISON["cer"]; pcm = PARAM_COUNTS["model"]
best_l = RATE_TABLES["test"].loc[[LANG_NAME[l] for l in LANGS], "WER %"].idxmin(); worst_l = RATE_TABLES["test"].loc[[LANG_NAME[l] for l in LANGS], "WER %"].idxmax()
vcur = RUNS[TAG]["val"]; v_ep1 = vcur[vcur.epoch <= 1.001].val_wer.min(); v_all = vcur.val_wer.min()
e2_gain = 100 * (v_ep1 - v_all)
bylen = LENGTH_TABLES["test"]["by_audio_duration"]; fail_all = FAIL_TABLE.query("split == 'test' and language == @POOLED").iloc[0]; sc_all = SCRIPT_TABLE.query("split == 'test' and language == @POOLED").iloc[0]
removed_ = tok_report[(tok_report.split == "test") & (tok_report["removal_%"] > 5)]; rep_ = stat_table["Repeat factor (instances / unique available)"]
direction = "lower" if rw["abs_diff_pts"] < 0 else "higher"
lines = [f"QUESTION: Does function-preserving scaling (more trainable depth, wider encoder and decoder FFNs, SpecAugment, 2 epochs, best-checkpoint selection) improve a dense Whisper-small on eight Indic FLEURS languages?", "", "ANSWER (strictly from the measured results):",
         f"  {MODEL_NAME} reached a pooled test WER of {rw['new']:.2f} % (95 % cluster-bootstrap CI [{rw['ci95'][0]:.2f}, {rw['ci95'][1]:.2f}]) and CER {rc['new']:.2f} % (CI [{rc['ci95'][0]:.2f}, {rc['ci95'][1]:.2f}]); "
         f"the baseline Dense-198M (single run) had {rw['previous']:.2f} % / {rc['previous']:.2f} %, i.e. {abs(rw['abs_diff_pts']):.2f} pts {direction} WER ({rw['rel_change_pct']:+.1f} % relative) and {abs(rc['abs_diff_pts']):.2f} pts {'lower' if rc['abs_diff_pts'] < 0 else 'higher'} CER "
         f"({rc['rel_change_pct']:+.1f} % relative). The baseline WER value {'lies inside' if rw['previous_inside_new_ci'] else 'lies outside'} the new 95 % interval.", "",
         f"1. ARCHITECTURE: {MODEL_NAME}: {pcm['trainable']:,} trainable / {pcm['total']:,} total parameters (x{REFERENCE_COMPARISON['trainable_params_ratio']:.2f} the trainable parameters of the baseline); encoder layers {[i + 1 for i in CFG['TRAIN_ENC_LAYERS']]} and decoder layers {[i + 1 for i in CFG['TRAIN_DEC_LAYERS']]} trainable, FFNs widened (encoder 768 -> {ENC_FFN_WIDTH} -> 768, decoder 768 -> {DEC_FFN_WIDTH} -> 768; function-preserving, initial equivalence verified: max logit error {EQUIV_RESULT['model_logit_max_abs_err']:.1e}).",
         f"2. TRAINING: {RUNS[TAG]['steps']}/{TOTAL_STEPS} optimiser steps ({RUNS[TAG]['steps'] / STEPS_PER_EPOCH:.2f} of {CFG['NUM_EPOCHS']} epochs){' - early-stopped' if RUNS[TAG].get('early_stopped') else ''}; selected checkpoint: step {SELECTED_STEP} (epoch {SELECTED_STEP / STEPS_PER_EPOCH:.2f}), chosen on the FULL validation set among the best {CFG['CKPT_TOPK']} subset checkpoints; "
         f"training time {RUNS[TAG]['train_seconds']/60:.1f} min (baseline {CFG['PREVIOUS_DENSE_198M']['train_minutes']:.0f} min, x{REFERENCE_COMPARISON['train_time_ratio']:.2f}); peak GPU memory {RUNS[TAG]['peak_gpu_mem_gb']:.1f} GB; OOM retries {RUNS[TAG]['oom_events']}; skipped fp16 steps {RUNS[TAG]['skipped_optimizer_steps']}. "
         f"Validation-subset WER: best within epoch 1 {100 * v_ep1:.2f} %, best overall {100 * v_all:.2f} % ({e2_gain:+.2f} pts from the second epoch) - " + ("the second epoch helped on the validation subset." if e2_gain > 0.2 else "the second epoch did not help measurably on the validation subset."),
         f"3. PER-LANGUAGE (test WER %): " + "; ".join(f"{LANG_NAME[l]} {RATE_TABLES['test'].loc[LANG_NAME[l], 'WER %']:.1f}" for l in LANGS) + f". Best: {best_l}; worst: {worst_l}. Macro-average WER {macro['WER %']:.2f} % vs pooled {pooled['WER %']:.2f} %.",
         f"4. WER BY UTTERANCE LENGTH (test, audio-duration quintiles): " + ", ".join(f"{m_:.1f}s: {w_:.1f} %" for m_, w_ in zip(bylen.mean_audio_s, bylen['WER %'])) + ".",
         f"5. FAILURES / SCRIPT (test, pooled): failure rate {100 * fail_all.failure_rate:.1f} % (categories overlap; 'excessive' is a heuristic), script accuracy {sc_all.correct_script_pct:.1f} % (predominant expected-script rate).",
         f"6. TOKENISATION: Spearman rho(tokens/char, WER) = {SPEARMAN['tokens_per_char vs WER%']['rho']:+.2f} (p = {SPEARMAN['tokens_per_char vs WER%']['p']:.2f}, n = 8 languages; descriptive).",
         f"7. INFERENCE: RTF {sp1.rtf:.4f} (batch 1) / {sp16.rtf:.4f} (batch 16), {sp16.tokens_per_sec:.0f} tokens/s (batch 16); model only, features pre-extracted.",
         "8. LIMITATIONS: one training run / one seed (seed-to-seed variance is not captured); the improvement over the baseline bundles several changes and cannot be attributed to any single one (no ablations were run, by design); the baseline is a single earlier run without per-utterance outputs, so the comparison is descriptive, not paired; "
         f"FLEURS is small (~{int(st_av.n.sum()):,} unique clean training utterances, {float(st_av.hours.sum()):.0f} h) and the balanced pool REPEATS utterances (instances / unique available: " + ", ".join(f"{LANG_NAME[l]} {rep_[l]:.2f}x" for l in LANGS) + "); "
         "the 440-token cap removes a language-dependent share of utterances" + (" (>5 % of the test set for: " + ", ".join(f"{LANG_NAME[r.lang]} {r['removal_%']:.1f} %" for _, r in removed_.iterrows()) + ")" if len(removed_) else "") +
         "; evaluation uses a script-aware normaliser (not comparable with Whisper-BasicTextNormalizer leaderboards); the trainable set is large relative to the amount of unique training speech, so over-fitting is controlled only by SpecAugment, checkpoint selection and early stopping; no claim beyond this recipe on FLEURS is made."]
SUMMARY = "\n".join(lines); print(SUMMARY)
with open(f"{OUT_DIR}/research_summary.txt", "w", encoding="utf-8") as f: f.write(SUMMARY)
results.update(research_summary=SUMMARY, notebook_runtime_seconds=time.time() - NOTEBOOK_T0); write_results()
print_final_configuration()